In [ ]:
import pandas as pd
import numpy as np
from google.colab import files

In [ ]:
from scipy import stats

def _mean_ci(vals, conf=0.95):
    vals = np.asarray(vals, dtype=float)
    vals = vals[~np.isnan(vals)]
    n = len(vals)
    if n < 2:
        return (np.mean(vals) if n else np.nan), np.nan, np.nan
    m  = np.mean(vals)
    se = stats.sem(vals)
    h  = se * stats.t.ppf((1 + conf) / 2., n - 1)
    return m, m - h, m + h

def three_way_table(model_name, y_train, train_pred, fold_results, y_test, test_pred, n_boot=1000, seed=42):
    """
    In-Sample Fit vs Validation Folds vs Test Forecasts — three procedures,
    clearly separated, instead of comparing across mismatched ones.
    fold_results: the model's *_fvl / *_fold_val_res list (dicts with
    'mae', 'mape', 'r2' per fold, already computed by the walk-forward loop).
    """
    rng = np.random.default_rng(seed)

    def _metrics(y_t, y_p):
        y_t, y_p = np.asarray(y_t).ravel(), np.asarray(y_p).ravel()
        mae  = mean_absolute_error(y_t, y_p)
        rmse = np.sqrt(mean_squared_error(y_t, y_p))
        r2   = r2_score(y_t, y_p)
        nz   = y_t != 0
        mape = np.mean(np.abs((y_t[nz]-y_p[nz])/y_t[nz]))*100
        return mae, rmse, r2, mape

    def _bootstrap_point(y_t, y_p, n_boot):
        y_t, y_p = np.asarray(y_t).ravel(), np.asarray(y_p).ravel()
        n = len(y_t)
        maes, mapes = [], []
        for _ in range(n_boot):
            idx = rng.integers(0, n, n)
            yt, yp = y_t[idx], y_p[idx]
            maes.append(mean_absolute_error(yt, yp))
            nz = yt != 0
            mapes.append(np.mean(np.abs((yt[nz]-yp[nz])/yt[nz]))*100 if nz.any() else np.nan)
        return (np.nanmean(maes), np.nanpercentile(maes,2.5), np.nanpercentile(maes,97.5)), \
               (np.nanmean(mapes), np.nanpercentile(mapes,2.5), np.nanpercentile(mapes,97.5))

    train_mae, train_rmse, train_r2, train_mape = _metrics(y_train, train_pred)

    fold_df = pd.DataFrame(fold_results)
    val_mae_m,  *_ = _mean_ci(fold_df['mae'])
    val_mape_m, val_mape_lo, val_mape_hi = _mean_ci(fold_df['mape'])
    val_r2_m,   *_ = _mean_ci(fold_df['r2'])

    (test_mae_m, test_mae_lo, test_mae_hi), (test_mape_m, test_mape_lo, test_mape_hi) = _bootstrap_point(y_test, test_pred, n_boot)
    _, _, test_r2, _ = _metrics(y_test, test_pred)

    print("="*78)
    print(f"{model_name.upper()} — IN-SAMPLE FIT vs VALIDATION FOLDS vs TEST FORECASTS")
    print("="*78)
    print(f"{'Stage':<20}{'MAE':>16}{'MAPE':>18}")
    print("-"*78)
    print(f"{'In-Sample Fit':<20}{train_mae:>16.4f}{train_mape:>17.2f}%")
    print(f"{'Validation Folds':<20}{val_mae_m:>16.4f}{val_mape_m:>17.2f}%")
    print(f"{'Test Forecasts':<20}{test_mae_m:>16.4f}{test_mape_m:>17.2f}%")
    print("-"*78)
    print(f"In-sample fit vs Test MAPE gap:    {test_mape_m - train_mape:+.2f} pts  (MAE gap: {test_mae_m - train_mae:+.4f})")
    print(f"Validation folds vs Test MAPE gap: {test_mape_m - val_mape_m:+.2f} pts  (MAE gap: {test_mae_m - val_mae_m:+.4f})")
    print("="*78)

    return {'model': model_name, 'train_mae': train_mae, 'train_mape': train_mape, 'train_r2': train_r2,
            'val_mae': val_mae_m, 'val_mape': val_mape_m, 'val_r2': val_r2_m,
            'test_mae': test_mae_m, 'test_mape': test_mape_m, 'test_r2': test_r2}

In [ ]:
stock_dortmund = pd.read_csv("Borussia Dortmund Stock Price History.csv")
stock_dortmund["Date"] = pd.to_datetime(stock_dortmund["Date"])

stock_dortmund

In [ ]:
match_dortmund = pd.read_csv("dortmund_2000_to_2025.csv")

match_dortmund = match_dortmund[["hometeam", "awayteam", "homeelo", "awayelo", "ftresult", "matchdate"]]
match_dortmund["Date"] = pd.to_datetime(match_dortmund["matchdate"])

df_filtered = pd.DataFrame()

In [ ]:
#Date
df_filtered['Date'] = match_dortmund['Date']
#Result
dortmund_result = []
for i in range(len(match_dortmund)):
    if (match_dortmund['ftresult'].iloc[i] == 'A' and match_dortmund['hometeam'].iloc[i] == 'Dortmund') or \
       (match_dortmund['ftresult'].iloc[i] == 'H' and match_dortmund['awayteam'].iloc[i] == 'Dortmund'):
        dortmund_result.append(0)
    elif match_dortmund['ftresult'].iloc[i] == 'D':
        dortmund_result.append(1)
    else:
        dortmund_result.append(2)
df_filtered['dortmund_result'] = dortmund_result
#Elo
dortmund_elo = []
for i in range(len(match_dortmund)):
    if match_dortmund['hometeam'].iloc[i] == 'Dortmund':
        dortmund_elo.append(match_dortmund['homeelo'].iloc[i])
    else:
        dortmund_elo.append(match_dortmund['awayelo'].iloc[i])
df_filtered['dortmund_elo'] = dortmund_elo

df_filtered

In [ ]:
# Make sure both Date columns are just dates without time
df_filtered['Date'] = pd.to_datetime(df_filtered['Date']).dt.date
stock_dortmund['Date'] = pd.to_datetime(stock_dortmund['Date']).dt.date
min_date = stock_dortmund['Date'].min()
max_date = max(df_filtered['Date'].max(), stock_dortmund['Date'].max())
all_dates = pd.date_range(start=min_date, end=max_date, freq='D')
combined_df = pd.DataFrame({'Date': all_dates.date})
combined_df = combined_df.merge(df_filtered, on='Date', how='left')
combined_df = combined_df.merge(stock_dortmund, on='Date', how='left')
combined_df['Price'] = combined_df['Price'].ffill()

df_clean = combined_df.dropna(subset=['dortmund_result']).reset_index(drop=True)

print(f"df_clean shape: {df_clean.shape}")
combined_df.head(60)

In [ ]:
# ============================================================================
# 20 representative match-date / price-date pairs
# Reconstructs the actual trading day each match's price came from, since
# ffill() overwrites the Price column without keeping the source date.
# ============================================================================
match_dates_df = df_clean[['Date']].copy()
match_dates_df['Date'] = pd.to_datetime(match_dates_df['Date'])

raw_prices = stock_dortmund.copy()
raw_prices['Date'] = pd.to_datetime(raw_prices['Date'])
raw_prices = raw_prices.sort_values('Date')

pairs = pd.merge_asof(
    match_dates_df.sort_values('Date'),
    raw_prices[['Date']].rename(columns={'Date': 'Price_Date'}),
    left_on='Date', right_on='Price_Date',
    direction='backward'
)
pairs = pairs.rename(columns={'Date': 'Match_Date'})

# Evenly spaced sample of 20 matches across the full dataset
sample_idx = np.linspace(0, len(pairs) - 1, 20, dtype=int)
sample = pairs.iloc[sample_idx].reset_index(drop=True)

print("20 REPRESENTATIVE MATCH-DATE / PRICE-DATE PAIRS")
print("="*50)
print(sample.to_string(index=False))

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt

In [ ]:
print("="*70)
print("CREATING LAGGED FEATURES")
print("="*70)

df_clean = df_clean.sort_values('Date').reset_index(drop=True)

lag_columns = []
n_lags = 4

for i in range(1, n_lags + 1):
    col_name = f'result_lag{i}'
    df_clean[col_name] = df_clean['dortmund_result'].shift(i)
    lag_columns.append(col_name)

for i in range(1, n_lags + 1):
    col_name = f'elo_lag{i}'
    df_clean[col_name] = df_clean['dortmund_elo'].shift(i)
    lag_columns.append(col_name)

for i in range(1, n_lags + 1):
    col_name = f'price_lag{i}'
    df_clean[col_name] = df_clean['Price'].shift(i)
    lag_columns.append(col_name)

df_model = df_clean.dropna(subset=lag_columns).reset_index(drop=True)

print(f"\nRows before lag-window trimming: {len(df_clean)}")
print(f"Rows after lag-window trimming: {len(df_model)}")
print(f"Rows dropped for incomplete lag history: {len(df_clean) - len(df_model)}")
print(f"\nDataset shape: {df_model.shape}")
print(f"Features created: {lag_columns}")

In [ ]:
X = df_model[lag_columns]
y = df_model['Price']

print("\n" + "="*70)
print("TRAIN/TEST SPLIT")
print("="*70)
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Features (X): {list(X.columns)}")
print(f"Target (y): Price")

train_size = int(0.8 * len(X))
X_train = X.iloc[:train_size]
X_test  = X.iloc[train_size:]
y_train = y.iloc[:train_size]
y_test  = y.iloc[train_size:]

print(f"\nTrain set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")

In [ ]:
# ============================================================================
# Cell 4: GridSearch with TimeSeriesSplit (UPDATED)
# ============================================================================

print("\n" + "="*70)
print("GRIDSEARCH - RANDOM FOREST")
print("="*70)

param_grid = {
    'n_estimators': [100, 500],
    'max_features': list(range(2, X_train.shape[1], 2)),
    'min_samples_split': list(range(20, 100, 10))
}

print(f"Parameter grid:")
print(f"  n_estimators: {param_grid['n_estimators']}")
print(f"  max_features: {param_grid['max_features']}")
print(f"  min_samples_split: {param_grid['min_samples_split']}")

total_combinations = (len(param_grid['n_estimators']) *
                      len(param_grid['max_features']) *
                      len(param_grid['min_samples_split']))
print(f"\nTotal combinations: {total_combinations:,}")

tscv = TimeSeriesSplit(n_splits=10)
rf_base = RandomForestRegressor(random_state=42, n_jobs=-1)

grid_search = GridSearchCV(
    estimator=rf_base,
    param_grid=param_grid,
    cv=tscv,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=4,
    return_train_score=True
)

print(f"\nCross-validation: TimeSeriesSplit(n_splits=10)")
print(f"Scoring: neg_mean_absolute_error")
print("Starting GridSearch...\n")

grid_search.fit(X_train, y_train)

print("\n" + "="*70)
print("GRIDSEARCH COMPLETE")
print("="*70)
print(f"Best CV MAE: {-grid_search.best_score_:.4f}")
print(f"\nBest Parameters:")
for param, value in grid_search.best_params_.items():
    print(f"  {param}: {value}")

# ============================================================================
# Cell 5: Train Final Model and Evaluate
# ============================================================================

print("\n" + "="*70)
print("TRAINING FINAL MODEL WITH BEST PARAMETERS")
print("="*70)

final_rf = RandomForestRegressor(**grid_search.best_params_, random_state=42, n_jobs=-1)
final_rf.fit(X_train, y_train)

rf_train_pred = final_rf.predict(X_train)
rf_test_pred  = final_rf.predict(X_test)

rf_train_r2   = r2_score(y_train, rf_train_pred)
rf_train_mae  = mean_absolute_error(y_train, rf_train_pred)
rf_train_rmse = np.sqrt(mean_squared_error(y_train, rf_train_pred))
rf_train_mape = np.mean(np.abs((y_train - rf_train_pred) / y_train)) * 100

rf_test_r2   = r2_score(y_test, rf_test_pred)
rf_test_mae  = mean_absolute_error(y_test, rf_test_pred)
rf_test_rmse = np.sqrt(mean_squared_error(y_test, rf_test_pred))
rf_test_mape = np.mean(np.abs((y_test - rf_test_pred) / y_test)) * 100

print("\n" + "="*70)
print("RANDOM FOREST - OVERALL RESULTS")
print("="*70)
print(f"{'Metric':<10} {'Train':>12} {'Test':>12}")
print("-"*36)
print(f"{'R²':<10} {rf_train_r2:>12.4f} {rf_test_r2:>12.4f}")
print(f"{'MAE':<10} {rf_train_mae:>12.4f} {rf_test_mae:>12.4f}")
print(f"{'RMSE':<10} {rf_train_rmse:>12.4f} {rf_test_rmse:>12.4f}")
print(f"{'MAPE':<10} {rf_train_mape:>11.2f}% {rf_test_mape:>11.2f}%")
print("="*70)

In [ ]:
import numpy as np
from sklearn.utils import resample
from sklearn.metrics import r2_score, mean_absolute_error

def bootstrap_ci(y_true, y_pred, n_iterations=1000, block_length=8, seed=42):
    """
    Moving-block bootstrap. Resamples contiguous blocks of `block_length`
    consecutive observations (with replacement) and concatenates them to
    rebuild a resampled series of the original length, preserving the
    autocorrelation between adjacent forecast errors that i.i.d. row
    resampling destroys.
    """
    rng = np.random.default_rng(seed)
    y_true = np.array(y_true).ravel()
    y_pred = np.array(y_pred).ravel()
    mask_nan = ~np.isnan(y_pred)
    y_true = y_true[mask_nan]
    y_pred = y_pred[mask_nan]
    n = len(y_true)
    n_blocks = int(np.ceil(n / block_length))

    st = {'r2': [], 'mae': [], 'mape': []}
    for _ in range(n_iterations):
        starts = rng.integers(0, n - block_length + 1, n_blocks)
        idx = np.concatenate([np.arange(s, s + block_length) for s in starts])[:n]
        y_t, y_p = y_true[idx], y_pred[idx]
        st['r2'].append(r2_score(y_t, y_p))
        st['mae'].append(mean_absolute_error(y_t, y_p))
        nz = y_t != 0
        st['mape'].append(np.mean(np.abs((y_t[nz]-y_p[nz])/y_t[nz]))*100 if nz.any() else np.nan)
    return st

# --- TRAINING SET ---
_bt = bootstrap_ci(y_train, rf_train_pred)
print("="*70)
print("RANDOM FOREST - 95% BOOTSTRAP CI (TRAINING SET)")
print("="*70)
print(f"{'Metric':<8} {'Mean':>10} {'CI Lower':>12} {'CI Upper':>12}")
print("-"*44)
for m in ['r2', 'mae', 'mape']:
    mv = np.nanmean(_bt[m])
    lo = np.nanpercentile(_bt[m], 2.5)
    hi = np.nanpercentile(_bt[m], 97.5)
    lb = m.upper()
    if m == 'mape':
        print(f"{lb:<8} {mv:>9.2f}% {lo:>11.2f}% {hi:>11.2f}%")
    else:
        print(f"{lb:<8} {mv:>10.4f} {lo:>12.4f} {hi:>12.4f}")
print("="*70)

# --- TEST SET ---
_bt2 = bootstrap_ci(y_test, rf_test_pred)
print()
print("="*70)
print("RANDOM FOREST - 95% BOOTSTRAP CI (TEST SET)")
print("="*70)
print(f"{'Metric':<8} {'Mean':>10} {'CI Lower':>12} {'CI Upper':>12}")
print("-"*44)
for m in ['r2', 'mae', 'mape']:
    mv = np.nanmean(_bt2[m])
    lo = np.nanpercentile(_bt2[m], 2.5)
    hi = np.nanpercentile(_bt2[m], 97.5)
    lb = m.upper()
    if m == 'mape':
        print(f"{lb:<8} {mv:>9.2f}% {lo:>11.2f}% {hi:>11.2f}%")
    else:
        print(f"{lb:<8} {mv:>10.4f} {lo:>12.4f} {hi:>12.4f}")
print("="*70)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import r2_score, mean_absolute_error
from scipy import stats

def mean_ci(vals, conf=0.95):
    vals = np.asarray(vals, dtype=float)
    vals = vals[~np.isnan(vals)]
    n = len(vals)
    if n < 2:
        return np.mean(vals), np.nan, np.nan
    m  = np.mean(vals)
    se = stats.sem(vals)
    h  = se * stats.t.ppf((1 + conf) / 2., n - 1)
    return m, m - h, m + h

_tscv = TimeSeriesSplit(n_splits=10)
_fvl  = []

for fold, (tri, vli) in enumerate(_tscv.split(X_train), 1):
    _Xtr, _Xvl = X_train.iloc[tri], X_train.iloc[vli]
    _ytr, _yvl = y_train.iloc[tri], y_train.iloc[vli]
    _md = RandomForestRegressor(**grid_search.best_params_, random_state=42, n_jobs=-1)
    _md.fit(_Xtr, _ytr)
    _vp  = _md.predict(_Xvl)
    _yv  = np.array(_yvl).flatten()
    _mv  = _yv != 0
    _fvl.append({'fold': fold,
                 'r2':   r2_score(_yv, _vp),
                 'mae':  mean_absolute_error(_yv, _vp),
                 'mape': np.mean(np.abs((_yv[_mv]-_vp[_mv])/_yv[_mv]))*100})
    print(f"  Fold {fold} — Val MAE: {_fvl[-1]['mae']:.4f}")

print()
print("="*70)
print("RANDOM FOREST - 95% FOLD-LEVEL CI (VALIDATION)")
print("="*70)
print(f"{'Metric':<8} {'Mean':>10} {'CI Lower':>12} {'CI Upper':>12}")
print("-"*44)
for m in ['r2', 'mae', 'mape']:
    mv, lo, hi = mean_ci(pd.DataFrame(_fvl)[m])
    lb = m.upper()
    if m == 'mape':
        print(f"{lb:<8} {mv:>9.2f}% {lo:>11.2f}% {hi:>11.2f}%")
    else:
        print(f"{lb:<8} {mv:>10.4f} {lo:>12.4f} {hi:>12.4f}")
print("="*70)

rf_fvl = _fvl.copy()
rf_summary   = three_way_table('Random Forest', y_train, rf_train_pred, rf_fvl, y_test, rf_test_pred)

In [ ]:
# ============================================================================
# Cell 6: Feature Importance
# ============================================================================

feature_importance_rf = pd.DataFrame({
    'Feature': lag_columns,
    'Importance': final_rf.feature_importances_
}).sort_values('Importance', ascending=False)

print("\n" + "="*70)
print("FEATURE IMPORTANCE (Random Forest)")
print("="*70)
print(feature_importance_rf.to_string(index=False))

# ============================================================================
# Cell 7: Baseline Comparison
# ============================================================================

print("\n" + "="*70)
print("BASELINE MODEL (Predict Last Known Price)")
print("="*70)

baseline_pred = X_test['price_lag1'].values

baseline_r2   = r2_score(y_test, baseline_pred)
baseline_mae  = mean_absolute_error(y_test, baseline_pred)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))
baseline_mape = np.mean(np.abs((y_test - baseline_pred) / y_test)) * 100

print("\nBASELINE TEST METRICS")
print("-" * 70)
print(f"R² Score: {baseline_r2:.4f}")
print(f"MAE: {baseline_mae:.4f}")
print(f"RMSE: {baseline_rmse:.4f}")
print(f"MAPE: {baseline_mape:.2f}%")

print("\n" + "="*70)
print("COMPARISON: RANDOM FOREST vs BASELINE")
print("="*70)
print(f"RF R²:        {rf_test_r2:.4f}  |  Baseline R²:        {baseline_r2:.4f}")
print(f"RF MAE:       {rf_test_mae:.4f}  |  Baseline MAE:       {baseline_mae:.4f}")
print(f"RF MAPE:      {rf_test_mape:.2f}%  |  Baseline MAPE:      {baseline_mape:.2f}%")

if rf_test_r2 > baseline_r2:
    improvement = ((rf_test_r2 - baseline_r2) / abs(baseline_r2) * 100)
    print(f"\n✓ Random Forest is {improvement:.1f}% better than baseline!")
else:
    print(f"\n✗ Random Forest is worse than baseline.")

# ============================================================================
# Cell 8: Visualizations
# ============================================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

axes[0, 0].scatter(y_test, rf_test_pred, alpha=0.6, s=40, color='blue', edgecolors='black', linewidth=0.5)
axes[0, 0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2, label='Perfect Prediction')
axes[0, 0].set_title(f'Random Forest: Predicted vs Actual (Test)\nR²={rf_test_r2:.4f}, MAPE={rf_test_mape:.2f}%', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Actual Price', fontsize=12)
axes[0, 0].set_ylabel('Predicted Price', fontsize=12)
axes[0, 0].legend(fontsize=10)
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].scatter(y_test, baseline_pred, alpha=0.6, s=40, color='orange', edgecolors='black', linewidth=0.5)
axes[0, 1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2, label='Perfect Prediction')
axes[0, 1].set_title(f'Baseline: Predicted vs Actual (Test)\nR²={baseline_r2:.4f}, MAPE={baseline_mape:.2f}%', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Actual Price', fontsize=12)
axes[0, 1].set_ylabel('Predicted Price', fontsize=12)
axes[0, 1].legend(fontsize=10)
axes[0, 1].grid(True, alpha=0.3)

top_features = feature_importance_rf.head(12)
axes[1, 0].barh(range(len(top_features)), top_features['Importance'], color='steelblue', edgecolor='black')
axes[1, 0].set_yticks(range(len(top_features)))
axes[1, 0].set_yticklabels(top_features['Feature'])
axes[1, 0].set_xlabel('Importance', fontsize=12)
axes[1, 0].set_title('Random Forest: Feature Importance', fontsize=14, fontweight='bold')
axes[1, 0].invert_yaxis()
axes[1, 0].grid(True, alpha=0.3, axis='x')

axes[1, 1].bar(['Baseline', 'Random Forest'], [baseline_r2, rf_test_r2], color=['orange', 'blue'], alpha=0.7, edgecolor='black', linewidth=2)
axes[1, 1].set_ylabel('R² Score', fontsize=12)
axes[1, 1].set_title('Model Comparison (R² Score)', fontsize=14, fontweight='bold')
axes[1, 1].set_ylim([0, max(baseline_r2, rf_test_r2) * 1.15])
axes[1, 1].grid(True, alpha=0.3, axis='y')
for i, v in enumerate([baseline_r2, rf_test_r2]):
    axes[1, 1].text(i, v + 0.01, f'{v:.4f}', ha='center', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.show()

# ============================================================================
# Cell 9: Sample Predictions
# ============================================================================

print("\n" + "="*70)
print("SAMPLE PREDICTIONS (First 10 from Test Set)")
print("="*70)

sample_df = pd.DataFrame({
    'Actual':             y_test.values[:10],
    'RF_Predicted':       rf_test_pred[:10],
    'Baseline_Predicted': baseline_pred[:10],
    'RF_Error':           np.abs(y_test.values[:10] - rf_test_pred[:10]),
    'Baseline_Error':     np.abs(y_test.values[:10] - baseline_pred[:10])
})
print(sample_df.to_string(index=False))
print("\n✓ Random Forest model complete!")

In [ ]:
import shap
import matplotlib.pyplot as plt

best_rf   = grid_search.best_estimator_
explainer = shap.TreeExplainer(best_rf)

X_test_df   = X_test if not isinstance(X_test, np.ndarray) else pd.DataFrame(X_test, columns=lag_columns)
test_sample = X_test_df[-200:] if len(X_test_df) > 500 else X_test_df
shap_values = explainer(test_sample)

plt.figure(figsize=(10, 6))
shap.plots.bar(shap_values, show=False)
plt.title("RF Feature Importance (SHAP Bar)")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
shap.plots.beeswarm(shap_values, show=False)
plt.title("RF Feature Impact (Beeswarm)")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
shap.plots.heatmap(shap_values, show=False)
plt.title("RF Prediction Patterns (Heatmap)")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
shap.plots.waterfall(shap_values[-1], show=False)
plt.title("Local Explanation: Most Recent Prediction")
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

folds = [d['fold'] for d in _fvl]
maes  = [d['mape'] for d in _fvl]

# Approximate CI using ± std (or you can use your mean_ci function per fold)
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(folds, maes, color='orange', marker='o', linewidth=2, label='Random Forest')
ax.errorbar(folds, maes,
            yerr=[m * 0.3 for m in maes],  # placeholder spread — replace with actual CI if you have per-fold CI
            fmt='none', color='orange', alpha=0.5, capsize=4)

ax.set_xlabel('Fold')
ax.set_ylabel('MAPE (%)')
ax.set_title('Random Forest - MAPE Across CV Folds')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files

In [ ]:
stock_dortmund = pd.read_csv("Borussia Dortmund Stock Price History.csv")
stock_dortmund["Date"] = pd.to_datetime(stock_dortmund["Date"])

stock_dortmund

In [ ]:
match_dortmund = pd.read_csv("dortmund_2000_to_2025.csv")

match_dortmund = match_dortmund[["hometeam", "awayteam", "homeelo", "awayelo", "ftresult", "matchdate"]]
match_dortmund["Date"] = pd.to_datetime(match_dortmund["matchdate"])

df_filtered = pd.DataFrame()

In [ ]:
#Date
df_filtered['Date'] = match_dortmund['Date']
#Result
dortmund_result = []
for i in range(len(match_dortmund)):
    if (match_dortmund['ftresult'].iloc[i] == 'A' and match_dortmund['hometeam'].iloc[i] == 'Dortmund') or \
       (match_dortmund['ftresult'].iloc[i] == 'H' and match_dortmund['awayteam'].iloc[i] == 'Dortmund'):
        dortmund_result.append(0)
    elif match_dortmund['ftresult'].iloc[i] == 'D':
        dortmund_result.append(1)
    else:
        dortmund_result.append(2)
df_filtered['dortmund_result'] = dortmund_result
#Elo
dortmund_elo = []
for i in range(len(match_dortmund)):
    if match_dortmund['hometeam'].iloc[i] == 'Dortmund':
        dortmund_elo.append(match_dortmund['homeelo'].iloc[i])
    else:
        dortmund_elo.append(match_dortmund['awayelo'].iloc[i])
df_filtered['dortmund_elo'] = dortmund_elo

df_filtered

In [ ]:
df_filtered['Date'] = pd.to_datetime(df_filtered['Date']).dt.date
stock_dortmund['Date'] = pd.to_datetime(stock_dortmund['Date']).dt.date
min_date = stock_dortmund['Date'].min()
max_date = max(df_filtered['Date'].max(), stock_dortmund['Date'].max())
all_dates = pd.date_range(start=min_date, end=max_date, freq='D')
combined_df = pd.DataFrame({'Date': all_dates.date})
combined_df = combined_df.merge(df_filtered, on='Date', how='left')
combined_df = combined_df.merge(stock_dortmund, on='Date', how='left')
combined_df['Price'] = combined_df['Price'].ffill()

df_clean = combined_df.dropna(subset=['dortmund_result']).reset_index(drop=True)

print(f"df_clean shape: {df_clean.shape}")
combined_df.head(60)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt

In [ ]:
print("="*70)
print("CREATING LAGGED FEATURES")
print("="*70)

df_clean = df_clean.sort_values('Date').reset_index(drop=True)

lag_columns = []
n_lags = 4

for i in range(1, n_lags + 1):
    col_name = f'result_lag{i}'
    df_clean[col_name] = df_clean['dortmund_result'].shift(i)
    lag_columns.append(col_name)

for i in range(1, n_lags + 1):
    col_name = f'elo_lag{i}'
    df_clean[col_name] = df_clean['dortmund_elo'].shift(i)
    lag_columns.append(col_name)

for i in range(1, n_lags + 1):
    col_name = f'price_lag{i}'
    df_clean[col_name] = df_clean['Price'].shift(i)
    lag_columns.append(col_name)

df_model = df_clean.dropna(subset=lag_columns).reset_index(drop=True)

print(f"\nDataset shape: {df_model.shape}")
print(f"Features created: {lag_columns}")

In [ ]:
X = df_model[lag_columns]
y = df_model['Price']

print("\n" + "="*70)
print("TRAIN/TEST SPLIT")
print("="*70)
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Features (X): {list(X.columns)}")
print(f"Target (y): Price")

train_size = int(0.8 * len(X))
X_train = X.iloc[:train_size]
X_test  = X.iloc[train_size:]
y_train = y.iloc[:train_size]
y_test  = y.iloc[train_size:]

print(f"\nTrain set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")

In [ ]:
print("\n" + "="*70)
print("GRIDSEARCH - EXTRA TREES")
print("="*70)

param_grid = {
    'n_estimators': [100, 500],
    'max_features': list(range(2, X_train.shape[1], 2)),
    'min_samples_split': list(range(20, 100, 10))
}

print(f"Parameter grid:")
print(f"  n_estimators: {param_grid['n_estimators']}")
print(f"  max_features: {param_grid['max_features']}")
print(f"  min_samples_split: {param_grid['min_samples_split']}")

total_combinations = (len(param_grid['n_estimators']) *
                      len(param_grid['max_features']) *
                      len(param_grid['min_samples_split']))
print(f"\nTotal combinations: {total_combinations:,}")

tscv    = TimeSeriesSplit(n_splits=10)
et_base = ExtraTreesRegressor(random_state=42, n_jobs=-1)

grid_search = GridSearchCV(
    estimator=et_base,
    param_grid=param_grid,
    cv=tscv,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=4,
    return_train_score=True
)

print(f"\nCross-validation: TimeSeriesSplit(n_splits=10)")
print(f"Scoring: neg_mean_absolute_error")
print("Starting GridSearch...\n")

grid_search.fit(X_train, y_train)

print("\n" + "="*70)
print("GRIDSEARCH COMPLETE")
print("="*70)
print(f"Best CV MAE: {-grid_search.best_score_:.4f}")
print(f"\nBest Parameters:")
for param, value in grid_search.best_params_.items():
    print(f"  {param}: {value}")

print("\n" + "="*70)
print("TRAINING FINAL MODEL WITH BEST PARAMETERS")
print("="*70)

final_et = ExtraTreesRegressor(**grid_search.best_params_, random_state=42, n_jobs=-1)
final_et.fit(X_train, y_train)

et_train_pred = final_et.predict(X_train)
et_test_pred  = final_et.predict(X_test)

et_train_r2   = r2_score(y_train, et_train_pred)
et_train_mae  = mean_absolute_error(y_train, et_train_pred)
et_train_rmse = np.sqrt(mean_squared_error(y_train, et_train_pred))
et_train_mape = np.mean(np.abs((y_train - et_train_pred) / y_train)) * 100

et_test_r2   = r2_score(y_test, et_test_pred)
et_test_mae  = mean_absolute_error(y_test, et_test_pred)
et_test_rmse = np.sqrt(mean_squared_error(y_test, et_test_pred))
et_test_mape = np.mean(np.abs((y_test - et_test_pred) / y_test)) * 100

print("\n" + "="*70)
print("EXTRA TREES - OVERALL RESULTS")
print("="*70)
print(f"{'Metric':<10} {'Train':>12} {'Test':>12}")
print("-"*36)
print(f"{'R²':<10} {et_train_r2:>12.4f} {et_test_r2:>12.4f}")
print(f"{'MAE':<10} {et_train_mae:>12.4f} {et_test_mae:>12.4f}")
print(f"{'RMSE':<10} {et_train_rmse:>12.4f} {et_test_rmse:>12.4f}")
print(f"{'MAPE':<10} {et_train_mape:>11.2f}% {et_test_mape:>11.2f}%")
print("="*70)

In [ ]:
import numpy as np
from sklearn.utils import resample
from sklearn.metrics import r2_score, mean_absolute_error

def bootstrap_ci(y_true, y_pred, n_iterations=1000, block_length=8, seed=42):
    """
    Moving-block bootstrap. Resamples contiguous blocks of `block_length`
    consecutive observations (with replacement) and concatenates them to
    rebuild a resampled series of the original length, preserving the
    autocorrelation between adjacent forecast errors that i.i.d. row
    resampling destroys.
    """
    rng = np.random.default_rng(seed)
    y_true = np.array(y_true).ravel()
    y_pred = np.array(y_pred).ravel()
    mask_nan = ~np.isnan(y_pred)
    y_true = y_true[mask_nan]
    y_pred = y_pred[mask_nan]
    n = len(y_true)
    n_blocks = int(np.ceil(n / block_length))

    st = {'r2': [], 'mae': [], 'mape': []}
    for _ in range(n_iterations):
        starts = rng.integers(0, n - block_length + 1, n_blocks)
        idx = np.concatenate([np.arange(s, s + block_length) for s in starts])[:n]
        y_t, y_p = y_true[idx], y_pred[idx]
        st['r2'].append(r2_score(y_t, y_p))
        st['mae'].append(mean_absolute_error(y_t, y_p))
        nz = y_t != 0
        st['mape'].append(np.mean(np.abs((y_t[nz]-y_p[nz])/y_t[nz]))*100 if nz.any() else np.nan)
    return st

_bt = bootstrap_ci(y_train, et_train_pred)
print("="*70)
print("EXTRA TREES - 95% BOOTSTRAP CI (TRAINING SET)")
print("="*70)
print(f"{'Metric':<8} {'Mean':>10} {'CI Lower':>12} {'CI Upper':>12}")
print("-"*44)
for m in ['r2', 'mae', 'mape']:
    mv = np.nanmean(_bt[m])
    lo = np.nanpercentile(_bt[m], 2.5)
    hi = np.nanpercentile(_bt[m], 97.5)
    lb = m.upper()
    if m == 'mape':
        print(f"{lb:<8} {mv:>9.2f}% {lo:>11.2f}% {hi:>11.2f}%")
    else:
        print(f"{lb:<8} {mv:>10.4f} {lo:>12.4f} {hi:>12.4f}")
print("="*70)

_bt2 = bootstrap_ci(y_test, et_test_pred)
print()
print("="*70)
print("EXTRA TREES - 95% BOOTSTRAP CI (TEST SET)")
print("="*70)
print(f"{'Metric':<8} {'Mean':>10} {'CI Lower':>12} {'CI Upper':>12}")
print("-"*44)
for m in ['r2', 'mae', 'mape']:
    mv = np.nanmean(_bt2[m])
    lo = np.nanpercentile(_bt2[m], 2.5)
    hi = np.nanpercentile(_bt2[m], 97.5)
    lb = m.upper()
    if m == 'mape':
        print(f"{lb:<8} {mv:>9.2f}% {lo:>11.2f}% {hi:>11.2f}%")
    else:
        print(f"{lb:<8} {mv:>10.4f} {lo:>12.4f} {hi:>12.4f}")
print("="*70)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import r2_score, mean_absolute_error
from scipy import stats

def mean_ci(vals, conf=0.95):
    vals = np.asarray(vals, dtype=float)
    vals = vals[~np.isnan(vals)]
    n = len(vals)
    if n < 2:
        return np.mean(vals), np.nan, np.nan
    m  = np.mean(vals)
    se = stats.sem(vals)
    h  = se * stats.t.ppf((1 + conf) / 2., n - 1)
    return m, m - h, m + h

_tscv = TimeSeriesSplit(n_splits=10)
_fvl  = []

for fold, (tri, vli) in enumerate(_tscv.split(X_train), 1):
    _Xtr, _Xvl = X_train.iloc[tri], X_train.iloc[vli]
    _ytr, _yvl = y_train.iloc[tri], y_train.iloc[vli]
    _md = ExtraTreesRegressor(**grid_search.best_params_, random_state=42, n_jobs=-1)
    _md.fit(_Xtr, _ytr)
    _vp  = _md.predict(_Xvl)
    _yv  = np.array(_yvl).flatten()
    _mv  = _yv != 0
    _fvl.append({'fold': fold,
                 'r2':   r2_score(_yv, _vp),
                 'mae':  mean_absolute_error(_yv, _vp),
                 'mape': np.mean(np.abs((_yv[_mv]-_vp[_mv])/_yv[_mv]))*100})
    print(f"  Fold {fold} — Val MAE: {_fvl[-1]['mae']:.4f}")

print()
print("="*70)
print("EXTRA TREES - 95% FOLD-LEVEL CI (VALIDATION)")
print("="*70)
print(f"{'Metric':<8} {'Mean':>10} {'CI Lower':>12} {'CI Upper':>12}")
print("-"*44)
for m in ['r2', 'mae', 'mape']:
    mv, lo, hi = mean_ci(pd.DataFrame(_fvl)[m])
    lb = m.upper()
    if m == 'mape':
        print(f"{lb:<8} {mv:>9.2f}% {lo:>11.2f}% {hi:>11.2f}%")
    else:
        print(f"{lb:<8} {mv:>10.4f} {lo:>12.4f} {hi:>12.4f}")
print("="*70)

et_fvl = _fvl.copy()
et_summary   = three_way_table('Extra Trees', y_train, et_train_pred, et_fvl, y_test, et_test_pred)

In [ ]:
feature_importance_et = pd.DataFrame({
    'Feature': lag_columns,
    'Importance': final_et.feature_importances_
}).sort_values('Importance', ascending=False)

print("\n" + "="*70)
print("FEATURE IMPORTANCE (Extra Trees)")
print("="*70)
print(feature_importance_et.to_string(index=False))

print("\n" + "="*70)
print("BASELINE MODEL (Predict Last Known Price)")
print("="*70)

baseline_pred = X_test['price_lag1'].values
baseline_r2   = r2_score(y_test, baseline_pred)
baseline_mae  = mean_absolute_error(y_test, baseline_pred)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))
baseline_mape = np.mean(np.abs((y_test - baseline_pred) / y_test)) * 100

print("\nBASELINE TEST METRICS")
print("-" * 70)
print(f"R² Score: {baseline_r2:.4f}")
print(f"MAE: {baseline_mae:.4f}")
print(f"RMSE: {baseline_rmse:.4f}")
print(f"MAPE: {baseline_mape:.2f}%")

print("\n" + "="*70)
print("COMPARISON: EXTRA TREES vs BASELINE")
print("="*70)
print(f"ET R²:        {et_test_r2:.4f}  |  Baseline R²:        {baseline_r2:.4f}")
print(f"ET MAE:       {et_test_mae:.4f}  |  Baseline MAE:       {baseline_mae:.4f}")
print(f"ET MAPE:      {et_test_mape:.2f}%  |  Baseline MAPE:      {baseline_mape:.2f}%")

if et_test_r2 > baseline_r2:
    improvement = ((et_test_r2 - baseline_r2) / abs(baseline_r2) * 100)
    print(f"\n✓ Extra Trees is {improvement:.1f}% better than baseline!")
else:
    print(f"\n✗ Extra Trees is worse than baseline.")

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

axes[0, 0].scatter(y_test, et_test_pred, alpha=0.6, s=40, color='purple', edgecolors='black', linewidth=0.5)
axes[0, 0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2, label='Perfect Prediction')
axes[0, 0].set_title(f'Extra Trees: Predicted vs Actual (Test)\nR²={et_test_r2:.4f}, MAPE={et_test_mape:.2f}%', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Actual Price', fontsize=12)
axes[0, 0].set_ylabel('Predicted Price', fontsize=12)
axes[0, 0].legend(fontsize=10)
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].scatter(y_test, baseline_pred, alpha=0.6, s=40, color='orange', edgecolors='black', linewidth=0.5)
axes[0, 1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2, label='Perfect Prediction')
axes[0, 1].set_title(f'Baseline: Predicted vs Actual (Test)\nR²={baseline_r2:.4f}, MAPE={baseline_mape:.2f}%', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Actual Price', fontsize=12)
axes[0, 1].set_ylabel('Predicted Price', fontsize=12)
axes[0, 1].legend(fontsize=10)
axes[0, 1].grid(True, alpha=0.3)

top_features = feature_importance_et.head(12)
axes[1, 0].barh(range(len(top_features)), top_features['Importance'], color='mediumpurple', edgecolor='black')
axes[1, 0].set_yticks(range(len(top_features)))
axes[1, 0].set_yticklabels(top_features['Feature'])
axes[1, 0].set_xlabel('Importance', fontsize=12)
axes[1, 0].set_title('Extra Trees: Feature Importance', fontsize=14, fontweight='bold')
axes[1, 0].invert_yaxis()
axes[1, 0].grid(True, alpha=0.3, axis='x')

axes[1, 1].bar(['Baseline', 'Extra Trees'], [baseline_r2, et_test_r2], color=['orange', 'purple'], alpha=0.7, edgecolor='black', linewidth=2)
axes[1, 1].set_ylabel('R² Score', fontsize=12)
axes[1, 1].set_title('Model Comparison (R² Score)', fontsize=14, fontweight='bold')
axes[1, 1].set_ylim([0, max(baseline_r2, et_test_r2) * 1.15])
axes[1, 1].grid(True, alpha=0.3, axis='y')
for i, v in enumerate([baseline_r2, et_test_r2]):
    axes[1, 1].text(i, v + 0.01, f'{v:.4f}', ha='center', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("SAMPLE PREDICTIONS (First 10 from Test Set)")
print("="*70)

sample_df = pd.DataFrame({
    'Actual':             y_test.values[:10],
    'ET_Predicted':       et_test_pred[:10],
    'Baseline_Predicted': baseline_pred[:10],
    'ET_Error':           np.abs(y_test.values[:10] - et_test_pred[:10]),
    'Baseline_Error':     np.abs(y_test.values[:10] - baseline_pred[:10])
})
print(sample_df.to_string(index=False))
print("\n✓ Extra Trees model complete!")

In [ ]:
import shap
import matplotlib.pyplot as plt

best_et   = grid_search.best_estimator_
explainer = shap.TreeExplainer(best_et)

X_test_df   = X_test if not isinstance(X_test, np.ndarray) else pd.DataFrame(X_test, columns=lag_columns)
test_sample = X_test_df[-200:] if len(X_test_df) > 500 else X_test_df
shap_values = explainer(test_sample)

plt.figure(figsize=(10, 6))
shap.plots.bar(shap_values, show=False)
plt.title("ET Feature Importance (SHAP Bar)")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
shap.plots.beeswarm(shap_values, show=False)
plt.title("ET Feature Impact (Beeswarm)")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
shap.plots.heatmap(shap_values, show=False)
plt.title("ET Prediction Patterns (Heatmap)")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
shap.plots.waterfall(shap_values[-1], show=False)
plt.title("ET Local Explanation: Most Recent Prediction")
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files

In [ ]:
stock_dortmund = pd.read_csv("Borussia Dortmund Stock Price History.csv")
stock_dortmund["Date"] = pd.to_datetime(stock_dortmund["Date"])

stock_dortmund

In [ ]:
match_dortmund = pd.read_csv("dortmund_2000_to_2025.csv")

match_dortmund = match_dortmund[["hometeam", "awayteam", "homeelo", "awayelo", "ftresult", "matchdate"]]
match_dortmund["Date"] = pd.to_datetime(match_dortmund["matchdate"])

df_filtered = pd.DataFrame()

In [ ]:
#Date
df_filtered['Date'] = match_dortmund['Date']
#Result
dortmund_result = []
for i in range(len(match_dortmund)):
    if (match_dortmund['ftresult'].iloc[i] == 'A' and match_dortmund['hometeam'].iloc[i] == 'Dortmund') or \
       (match_dortmund['ftresult'].iloc[i] == 'H' and match_dortmund['awayteam'].iloc[i] == 'Dortmund'):
        dortmund_result.append(0)
    elif match_dortmund['ftresult'].iloc[i] == 'D':
        dortmund_result.append(1)
    else:
        dortmund_result.append(2)
df_filtered['dortmund_result'] = dortmund_result
#Elo
dortmund_elo = []
for i in range(len(match_dortmund)):
    if match_dortmund['hometeam'].iloc[i] == 'Dortmund':
        dortmund_elo.append(match_dortmund['homeelo'].iloc[i])
    else:
        dortmund_elo.append(match_dortmund['awayelo'].iloc[i])
df_filtered['dortmund_elo'] = dortmund_elo

df_filtered

In [ ]:
df_filtered['Date'] = pd.to_datetime(df_filtered['Date']).dt.date
stock_dortmund['Date'] = pd.to_datetime(stock_dortmund['Date']).dt.date
min_date = stock_dortmund['Date'].min()
max_date = max(df_filtered['Date'].max(), stock_dortmund['Date'].max())
all_dates = pd.date_range(start=min_date, end=max_date, freq='D')
combined_df = pd.DataFrame({'Date': all_dates.date})
combined_df = combined_df.merge(df_filtered, on='Date', how='left')
combined_df = combined_df.merge(stock_dortmund, on='Date', how='left')
combined_df['Price'] = combined_df['Price'].ffill()

df_clean = combined_df.dropna(subset=['dortmund_result']).reset_index(drop=True)

print(f"df_clean shape: {df_clean.shape}")
combined_df.head(60)

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt

In [ ]:
print("="*70)
print("CREATING LAGGED FEATURES")
print("="*70)

df_clean = df_clean.sort_values('Date').reset_index(drop=True)

lag_columns = []
n_lags = 4

for i in range(1, n_lags + 1):
    col_name = f'result_lag{i}'
    df_clean[col_name] = df_clean['dortmund_result'].shift(i)
    lag_columns.append(col_name)

for i in range(1, n_lags + 1):
    col_name = f'elo_lag{i}'
    df_clean[col_name] = df_clean['dortmund_elo'].shift(i)
    lag_columns.append(col_name)

for i in range(1, n_lags + 1):
    col_name = f'price_lag{i}'
    df_clean[col_name] = df_clean['Price'].shift(i)
    lag_columns.append(col_name)

df_model = df_clean.dropna(subset=lag_columns).reset_index(drop=True)

print(f"\nDataset shape: {df_model.shape}")
print(f"Features created: {lag_columns}")

In [ ]:
X = df_model[lag_columns]
y = df_model['Price']

print("\n" + "="*70)
print("TRAIN/TEST SPLIT")
print("="*70)
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Features (X): {list(X.columns)}")
print(f"Target (y): Price")

train_size = int(0.8 * len(X))
X_train = X.iloc[:train_size]
X_test  = X.iloc[train_size:]
y_train = y.iloc[:train_size]
y_test  = y.iloc[train_size:]

print(f"\nTrain set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")

In [ ]:
print("\n" + "="*70)
print("GRIDSEARCH - XGBOOST")
print("="*70)

param_grid = {
    'n_estimators':     [300, 500, 700],
    'subsample':        [0.4, 0.5, 0.6],
    'colsample_bynode': [0.4, 0.5, 0.6],
    'learning_rate':    [0.001, 0.003, 0.005, 0.01],
}

print(f"Parameter grid:")
print(f"  n_estimators: {param_grid['n_estimators']}")
print(f"  subsample: {param_grid['subsample']}")
print(f"  colsample_bynode: {param_grid['colsample_bynode']}")
print(f"  learning_rate: {param_grid['learning_rate']}")

total_combinations = (len(param_grid['n_estimators']) *
                      len(param_grid['subsample']) *
                      len(param_grid['colsample_bynode']) *
                      len(param_grid['learning_rate']))
print(f"\nTotal combinations: {total_combinations:,}")

tscv     = TimeSeriesSplit(n_splits=10)
xgb_base = XGBRegressor(random_state=42, n_jobs=-1)

grid_search = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid,
    cv=tscv,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=4,
    return_train_score=True
)

print(f"\nCross-validation: TimeSeriesSplit(n_splits=10)")
print(f"Scoring: neg_mean_absolute_error")
print("Starting GridSearch...\n")

grid_search.fit(X_train, y_train)

print("\n" + "="*70)
print("GRIDSEARCH COMPLETE")
print("="*70)
print(f"Best CV MAE: {-grid_search.best_score_:.4f}")
print(f"\nBest Parameters:")
for param, value in grid_search.best_params_.items():
    print(f"  {param}: {value}")

print("\n" + "="*70)
print("TRAINING FINAL MODEL WITH BEST PARAMETERS")
print("="*70)

final_xgb = XGBRegressor(**grid_search.best_params_, random_state=42, n_jobs=-1)
final_xgb.fit(X_train, y_train)

xgb_train_pred = final_xgb.predict(X_train)
xgb_test_pred  = final_xgb.predict(X_test)

xgb_train_r2   = r2_score(y_train, xgb_train_pred)
xgb_train_mae  = mean_absolute_error(y_train, xgb_train_pred)
xgb_train_rmse = np.sqrt(mean_squared_error(y_train, xgb_train_pred))
xgb_train_mape = np.mean(np.abs((y_train - xgb_train_pred) / y_train)) * 100

xgb_test_r2   = r2_score(y_test, xgb_test_pred)
xgb_test_mae  = mean_absolute_error(y_test, xgb_test_pred)
xgb_test_rmse = np.sqrt(mean_squared_error(y_test, xgb_test_pred))
xgb_test_mape = np.mean(np.abs((y_test - xgb_test_pred) / y_test)) * 100

print("\n" + "="*70)
print("XGBOOST - OVERALL RESULTS")
print("="*70)
print(f"{'Metric':<10} {'Train':>12} {'Test':>12}")
print("-"*36)
print(f"{'R²':<10} {xgb_train_r2:>12.4f} {xgb_test_r2:>12.4f}")
print(f"{'MAE':<10} {xgb_train_mae:>12.4f} {xgb_test_mae:>12.4f}")
print(f"{'RMSE':<10} {xgb_train_rmse:>12.4f} {xgb_test_rmse:>12.4f}")
print(f"{'MAPE':<10} {xgb_train_mape:>11.2f}% {xgb_test_mape:>11.2f}%")
print("="*70)

In [ ]:
import numpy as np
from sklearn.utils import resample
from sklearn.metrics import r2_score, mean_absolute_error

def bootstrap_ci(y_true, y_pred, n_iterations=1000, block_length=8, seed=42):
    """
    Moving-block bootstrap. Resamples contiguous blocks of `block_length`
    consecutive observations (with replacement) and concatenates them to
    rebuild a resampled series of the original length, preserving the
    autocorrelation between adjacent forecast errors that i.i.d. row
    resampling destroys.
    """
    rng = np.random.default_rng(seed)
    y_true = np.array(y_true).ravel()
    y_pred = np.array(y_pred).ravel()
    mask_nan = ~np.isnan(y_pred)
    y_true = y_true[mask_nan]
    y_pred = y_pred[mask_nan]
    n = len(y_true)
    n_blocks = int(np.ceil(n / block_length))

    st = {'r2': [], 'mae': [], 'mape': []}
    for _ in range(n_iterations):
        starts = rng.integers(0, n - block_length + 1, n_blocks)
        idx = np.concatenate([np.arange(s, s + block_length) for s in starts])[:n]
        y_t, y_p = y_true[idx], y_pred[idx]
        st['r2'].append(r2_score(y_t, y_p))
        st['mae'].append(mean_absolute_error(y_t, y_p))
        nz = y_t != 0
        st['mape'].append(np.mean(np.abs((y_t[nz]-y_p[nz])/y_t[nz]))*100 if nz.any() else np.nan)
    return st

_bt = bootstrap_ci(y_train, xgb_train_pred)
print("="*70)
print("XGBOOST - 95% BOOTSTRAP CI (TRAINING SET)")
print("="*70)
print(f"{'Metric':<8} {'Mean':>10} {'CI Lower':>12} {'CI Upper':>12}")
print("-"*44)
for m in ['r2', 'mae', 'mape']:
    mv = np.nanmean(_bt[m])
    lo = np.nanpercentile(_bt[m], 2.5)
    hi = np.nanpercentile(_bt[m], 97.5)
    lb = m.upper()
    if m == 'mape':
        print(f"{lb:<8} {mv:>9.2f}% {lo:>11.2f}% {hi:>11.2f}%")
    else:
        print(f"{lb:<8} {mv:>10.4f} {lo:>12.4f} {hi:>12.4f}")
print("="*70)

_bt2 = bootstrap_ci(y_test, xgb_test_pred)
print()
print("="*70)
print("XGBOOST - 95% BOOTSTRAP CI (TEST SET)")
print("="*70)
print(f"{'Metric':<8} {'Mean':>10} {'CI Lower':>12} {'CI Upper':>12}")
print("-"*44)
for m in ['r2', 'mae', 'mape']:
    mv = np.nanmean(_bt2[m])
    lo = np.nanpercentile(_bt2[m], 2.5)
    hi = np.nanpercentile(_bt2[m], 97.5)
    lb = m.upper()
    if m == 'mape':
        print(f"{lb:<8} {mv:>9.2f}% {lo:>11.2f}% {hi:>11.2f}%")
    else:
        print(f"{lb:<8} {mv:>10.4f} {lo:>12.4f} {hi:>12.4f}")
print("="*70)

In [ ]:
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import r2_score, mean_absolute_error
from scipy import stats

def mean_ci(vals, conf=0.95):
    vals = np.asarray(vals, dtype=float)
    vals = vals[~np.isnan(vals)]
    n = len(vals)
    if n < 2:
        return np.mean(vals), np.nan, np.nan
    m  = np.mean(vals)
    se = stats.sem(vals)
    h  = se * stats.t.ppf((1 + conf) / 2., n - 1)
    return m, m - h, m + h

_tscv = TimeSeriesSplit(n_splits=10)
_fvl  = []

for fold, (tri, vli) in enumerate(_tscv.split(X_train), 1):
    _Xtr, _Xvl = X_train.iloc[tri], X_train.iloc[vli]
    _ytr, _yvl = y_train.iloc[tri], y_train.iloc[vli]
    _md = XGBRegressor(**grid_search.best_params_, random_state=42, n_jobs=-1)
    _md.fit(_Xtr, _ytr)
    _vp  = _md.predict(_Xvl)
    _yv  = np.array(_yvl).flatten()
    _mv  = _yv != 0
    _fvl.append({'fold': fold,
                 'r2':   r2_score(_yv, _vp),
                 'mae':  mean_absolute_error(_yv, _vp),
                 'mape': np.mean(np.abs((_yv[_mv]-_vp[_mv])/_yv[_mv]))*100})
    print(f"  Fold {fold} — Val MAE: {_fvl[-1]['mae']:.4f}")

print()
print("="*70)
print("XGBOOST - 95% FOLD-LEVEL CI (VALIDATION)")
print("="*70)
print(f"{'Metric':<8} {'Mean':>10} {'CI Lower':>12} {'CI Upper':>12}")
print("-"*44)
for m in ['r2', 'mae', 'mape']:
    mv, lo, hi = mean_ci(pd.DataFrame(_fvl)[m])
    lb = m.upper()
    if m == 'mape':
        print(f"{lb:<8} {mv:>9.2f}% {lo:>11.2f}% {hi:>11.2f}%")
    else:
        print(f"{lb:<8} {mv:>10.4f} {lo:>12.4f} {hi:>12.4f}")
print("="*70)

xgb_fvl = _fvl.copy()
xgb_summary  = three_way_table('XGBoost', y_train, xgb_train_pred, xgb_fvl, y_test, xgb_test_pred)

In [ ]:
feature_importance_xgb = pd.DataFrame({
    'Feature': lag_columns,
    'Importance': final_xgb.feature_importances_
}).sort_values('Importance', ascending=False)

print("\n" + "="*70)
print("FEATURE IMPORTANCE (XGBoost)")
print("="*70)
print(feature_importance_xgb.to_string(index=False))

print("\n" + "="*70)
print("BASELINE MODEL (Predict Last Known Price)")
print("="*70)

baseline_pred = X_test['price_lag1'].values
baseline_r2   = r2_score(y_test, baseline_pred)
baseline_mae  = mean_absolute_error(y_test, baseline_pred)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))
baseline_mape = np.mean(np.abs((y_test - baseline_pred) / y_test)) * 100

print("\nBASELINE TEST METRICS")
print("-" * 70)
print(f"R² Score: {baseline_r2:.4f}")
print(f"MAE: {baseline_mae:.4f}")
print(f"RMSE: {baseline_rmse:.4f}")
print(f"MAPE: {baseline_mape:.2f}%")

print("\n" + "="*70)
print("COMPARISON: XGBOOST vs BASELINE")
print("="*70)
print(f"XGB R²:       {xgb_test_r2:.4f}  |  Baseline R²:       {baseline_r2:.4f}")
print(f"XGB MAE:      {xgb_test_mae:.4f}  |  Baseline MAE:      {baseline_mae:.4f}")
print(f"XGB MAPE:     {xgb_test_mape:.2f}%  |  Baseline MAPE:     {baseline_mape:.2f}%")

if xgb_test_r2 > baseline_r2:
    improvement = ((xgb_test_r2 - baseline_r2) / abs(baseline_r2) * 100)
    print(f"\n✓ XGBoost is {improvement:.1f}% better than baseline!")
else:
    print(f"\n✗ XGBoost is worse than baseline.")

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

axes[0, 0].scatter(y_test, xgb_test_pred, alpha=0.6, s=40, color='green', edgecolors='black', linewidth=0.5)
axes[0, 0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2, label='Perfect Prediction')
axes[0, 0].set_title(f'XGBoost: Predicted vs Actual (Test)\nR²={xgb_test_r2:.4f}, MAPE={xgb_test_mape:.2f}%', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Actual Price', fontsize=12)
axes[0, 0].set_ylabel('Predicted Price', fontsize=12)
axes[0, 0].legend(fontsize=10)
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].scatter(y_test, baseline_pred, alpha=0.6, s=40, color='orange', edgecolors='black', linewidth=0.5)
axes[0, 1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2, label='Perfect Prediction')
axes[0, 1].set_title(f'Baseline: Predicted vs Actual (Test)\nR²={baseline_r2:.4f}, MAPE={baseline_mape:.2f}%', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Actual Price', fontsize=12)
axes[0, 1].set_ylabel('Predicted Price', fontsize=12)
axes[0, 1].legend(fontsize=10)
axes[0, 1].grid(True, alpha=0.3)

top_features = feature_importance_xgb.head(12)
axes[1, 0].barh(range(len(top_features)), top_features['Importance'], color='lightgreen', edgecolor='black')
axes[1, 0].set_yticks(range(len(top_features)))
axes[1, 0].set_yticklabels(top_features['Feature'])
axes[1, 0].set_xlabel('Importance', fontsize=12)
axes[1, 0].set_title('XGBoost: Feature Importance', fontsize=14, fontweight='bold')
axes[1, 0].invert_yaxis()
axes[1, 0].grid(True, alpha=0.3, axis='x')

axes[1, 1].bar(['Baseline', 'XGBoost'], [baseline_r2, xgb_test_r2], color=['orange', 'green'], alpha=0.7, edgecolor='black', linewidth=2)
axes[1, 1].set_ylabel('R² Score', fontsize=12)
axes[1, 1].set_title('Model Comparison (R² Score)', fontsize=14, fontweight='bold')
axes[1, 1].set_ylim([0, max(baseline_r2, xgb_test_r2) * 1.15])
axes[1, 1].grid(True, alpha=0.3, axis='y')
for i, v in enumerate([baseline_r2, xgb_test_r2]):
    axes[1, 1].text(i, v + 0.01, f'{v:.4f}', ha='center', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("SAMPLE PREDICTIONS (First 10 from Test Set)")
print("="*70)

sample_df = pd.DataFrame({
    'Actual':             y_test.values[:10],
    'XGB_Predicted':      xgb_test_pred[:10],
    'Baseline_Predicted': baseline_pred[:10],
    'XGB_Error':          np.abs(y_test.values[:10] - xgb_test_pred[:10]),
    'Baseline_Error':     np.abs(y_test.values[:10] - baseline_pred[:10])
})
print(sample_df.to_string(index=False))
print("\n✓ XGBoost model complete!")

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files

In [ ]:
stock_dortmund = pd.read_csv("Borussia Dortmund Stock Price History.csv")
stock_dortmund["Date"] = pd.to_datetime(stock_dortmund["Date"])

stock_dortmund

In [ ]:
match_dortmund = pd.read_csv("dortmund_2000_to_2025.csv")

match_dortmund = match_dortmund[["hometeam", "awayteam", "homeelo", "awayelo", "ftresult", "matchdate"]]
match_dortmund["Date"] = pd.to_datetime(match_dortmund["matchdate"])

df_filtered = pd.DataFrame()

In [ ]:
#Date
df_filtered['Date'] = match_dortmund['Date']
#Result
dortmund_result = []
for i in range(len(match_dortmund)):
    if (match_dortmund['ftresult'].iloc[i] == 'A' and match_dortmund['hometeam'].iloc[i] == 'Dortmund') or \
       (match_dortmund['ftresult'].iloc[i] == 'H' and match_dortmund['awayteam'].iloc[i] == 'Dortmund'):
        dortmund_result.append(0)
    elif match_dortmund['ftresult'].iloc[i] == 'D':
        dortmund_result.append(1)
    else:
        dortmund_result.append(2)
df_filtered['dortmund_result'] = dortmund_result
#Elo
dortmund_elo = []
for i in range(len(match_dortmund)):
    if match_dortmund['hometeam'].iloc[i] == 'Dortmund':
        dortmund_elo.append(match_dortmund['homeelo'].iloc[i])
    else:
        dortmund_elo.append(match_dortmund['awayelo'].iloc[i])
df_filtered['dortmund_elo'] = dortmund_elo

df_filtered

In [ ]:
df_filtered['Date'] = pd.to_datetime(df_filtered['Date']).dt.date
stock_dortmund['Date'] = pd.to_datetime(stock_dortmund['Date']).dt.date
min_date = stock_dortmund['Date'].min()
max_date = max(df_filtered['Date'].max(), stock_dortmund['Date'].max())
all_dates = pd.date_range(start=min_date, end=max_date, freq='D')
combined_df = pd.DataFrame({'Date': all_dates.date})
combined_df = combined_df.merge(df_filtered, on='Date', how='left')
combined_df = combined_df.merge(stock_dortmund, on='Date', how='left')
combined_df['Price'] = combined_df['Price'].ffill()

df_clean = combined_df.dropna(subset=['dortmund_result']).reset_index(drop=True)

print(f"df_clean shape: {df_clean.shape}")
combined_df.head(60)

In [ ]:
import pandas as pd
import numpy as np
from lightgbm import LGBMRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from scipy import stats
import matplotlib.pyplot as plt

In [ ]:
print("="*70)
print("CREATING LAGGED FEATURES")
print("="*70)

df_clean = df_clean.sort_values('Date').reset_index(drop=True)

lag_columns = []
n_lags = 4

for i in range(1, n_lags + 1):
    col_name = f'result_lag{i}'
    df_clean[col_name] = df_clean['dortmund_result'].shift(i)
    lag_columns.append(col_name)

for i in range(1, n_lags + 1):
    col_name = f'elo_lag{i}'
    df_clean[col_name] = df_clean['dortmund_elo'].shift(i)
    lag_columns.append(col_name)

for i in range(1, n_lags + 1):
    col_name = f'price_lag{i}'
    df_clean[col_name] = df_clean['Price'].shift(i)
    lag_columns.append(col_name)

df_model = df_clean.dropna(subset=lag_columns).reset_index(drop=True)

print(f"\nDataset shape: {df_model.shape}")
print(f"Features created: {lag_columns}")

print("\n" + "="*70)
print("TARGET VARIABLE: PRICE")
print("="*70)
print(f"Price statistics:")
print(f"  Mean: {df_model['Price'].mean():.4f}")
print(f"  Std:  {df_model['Price'].std():.4f}")
print(f"  Min:  {df_model['Price'].min():.4f}")
print(f"  Max:  {df_model['Price'].max():.4f}")

In [ ]:
X = df_model[lag_columns]
y = df_model['Price']

print("\n" + "="*70)
print("DATASET PREPARED")
print("="*70)
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Features (X): {list(X.columns)}")
print(f"Target (y): Price")

print("\n" + "="*70)
print("GRIDSEARCH - LIGHTGBM")
print("="*70)

param_grid = {
    'n_estimators':     [300, 500, 700],
    'subsample':        [0.4, 0.5, 0.6],
    'colsample_bynode': [0.4, 0.5, 0.6],
    'learning_rate':    [0.001, 0.003, 0.005, 0.01],
}

print(f"Parameter grid:")
print(f"  n_estimators:     {param_grid['n_estimators']}")
print(f"  subsample:        {param_grid['subsample']}")
print(f"  colsample_bynode: {param_grid['colsample_bynode']}")
print(f"  learning_rate:    {param_grid['learning_rate']}")

tscv      = TimeSeriesSplit(n_splits=10)
lgbm_base = LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1)

grid_search = GridSearchCV(
    estimator=lgbm_base,
    param_grid=param_grid,
    cv=tscv,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=4,
    return_train_score=True
)

total_combinations = (len(param_grid['n_estimators']) *
                      len(param_grid['subsample']) *
                      len(param_grid['colsample_bynode']) *
                      len(param_grid['learning_rate']))

print(f"\nTotal combinations: {total_combinations:,}")
print(f"Total fits (combinations × folds): {total_combinations * 5:,}")
print("Starting GridSearch...\n")

grid_search.fit(X, y)

print("\n" + "="*70)
print("GRIDSEARCH COMPLETE")
print("="*70)
print(f"Best CV MAE: {-grid_search.best_score_:.4f}")
print(f"\nBest Parameters:")
for param, value in grid_search.best_params_.items():
    print(f"  {param}: {value}")

In [ ]:
from lightgbm import LGBMRegressor
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

train_size_lgbm = int(0.8 * len(X))
X_train_lgbm    = X.iloc[:train_size_lgbm]
X_test_lgbm     = X.iloc[train_size_lgbm:]
y_train_lgbm    = y.iloc[:train_size_lgbm]
y_test_lgbm     = y.iloc[train_size_lgbm:]

final_lgbm = LGBMRegressor(**grid_search.best_params_, random_state=42, n_jobs=-1, verbose=-1)
final_lgbm.fit(X_train_lgbm, y_train_lgbm)

lgbm_train_pred = final_lgbm.predict(X_train_lgbm)
lgbm_test_pred  = final_lgbm.predict(X_test_lgbm)

lgbm_train_r2   = r2_score(y_train_lgbm, lgbm_train_pred)
lgbm_train_mae  = mean_absolute_error(y_train_lgbm, lgbm_train_pred)
lgbm_train_rmse = np.sqrt(mean_squared_error(y_train_lgbm, lgbm_train_pred))
lgbm_train_mape = np.mean(np.abs((y_train_lgbm - lgbm_train_pred) / y_train_lgbm)) * 100

lgbm_test_r2   = r2_score(y_test_lgbm, lgbm_test_pred)
lgbm_test_mae  = mean_absolute_error(y_test_lgbm, lgbm_test_pred)
lgbm_test_rmse = np.sqrt(mean_squared_error(y_test_lgbm, lgbm_test_pred))
lgbm_test_mape = np.mean(np.abs((y_test_lgbm - lgbm_test_pred) / y_test_lgbm)) * 100

print("\n" + "="*70)
print("LIGHTGBM - OVERALL RESULTS")
print("="*70)
print(f"{'Metric':<10} {'Train':>12} {'Test':>12}")
print("-"*36)
print(f"{'R²':<10} {lgbm_train_r2:>12.4f} {lgbm_test_r2:>12.4f}")
print(f"{'MAE':<10} {lgbm_train_mae:>12.4f} {lgbm_test_mae:>12.4f}")
print(f"{'RMSE':<10} {lgbm_train_rmse:>12.4f} {lgbm_test_rmse:>12.4f}")
print(f"{'MAPE':<10} {lgbm_train_mape:>11.2f}% {lgbm_test_mape:>11.2f}%")
print("="*70)

In [ ]:
import numpy as np
from sklearn.utils import resample
from sklearn.metrics import r2_score, mean_absolute_error

def bootstrap_ci(y_true, y_pred, n_iterations=1000, block_length=8, seed=42):
    """
    Moving-block bootstrap. Resamples contiguous blocks of `block_length`
    consecutive observations (with replacement) and concatenates them to
    rebuild a resampled series of the original length, preserving the
    autocorrelation between adjacent forecast errors that i.i.d. row
    resampling destroys.
    """
    rng = np.random.default_rng(seed)
    y_true = np.array(y_true).ravel()
    y_pred = np.array(y_pred).ravel()
    mask_nan = ~np.isnan(y_pred)
    y_true = y_true[mask_nan]
    y_pred = y_pred[mask_nan]
    n = len(y_true)
    n_blocks = int(np.ceil(n / block_length))

    st = {'r2': [], 'mae': [], 'mape': []}
    for _ in range(n_iterations):
        starts = rng.integers(0, n - block_length + 1, n_blocks)
        idx = np.concatenate([np.arange(s, s + block_length) for s in starts])[:n]
        y_t, y_p = y_true[idx], y_pred[idx]
        st['r2'].append(r2_score(y_t, y_p))
        st['mae'].append(mean_absolute_error(y_t, y_p))
        nz = y_t != 0
        st['mape'].append(np.mean(np.abs((y_t[nz]-y_p[nz])/y_t[nz]))*100 if nz.any() else np.nan)
    return st

_bt = bootstrap_ci(y_train_lgbm, lgbm_train_pred)
print("="*70)
print("LIGHTGBM - 95% BOOTSTRAP CI (TRAINING SET)")
print("="*70)
print(f"{'Metric':<8} {'Mean':>10} {'CI Lower':>12} {'CI Upper':>12}")
print("-"*44)
for m in ['r2', 'mae', 'mape']:
    mv = np.nanmean(_bt[m])
    lo = np.nanpercentile(_bt[m], 2.5)
    hi = np.nanpercentile(_bt[m], 97.5)
    lb = m.upper()
    if m == 'mape':
        print(f"{lb:<8} {mv:>9.2f}% {lo:>11.2f}% {hi:>11.2f}%")
    else:
        print(f"{lb:<8} {mv:>10.4f} {lo:>12.4f} {hi:>12.4f}")
print("="*70)

_bt2 = bootstrap_ci(y_test_lgbm, lgbm_test_pred)
print()
print("="*70)
print("LIGHTGBM - 95% BOOTSTRAP CI (TEST SET)")
print("="*70)
print(f"{'Metric':<8} {'Mean':>10} {'CI Lower':>12} {'CI Upper':>12}")
print("-"*44)
for m in ['r2', 'mae', 'mape']:
    mv = np.nanmean(_bt2[m])
    lo = np.nanpercentile(_bt2[m], 2.5)
    hi = np.nanpercentile(_bt2[m], 97.5)
    lb = m.upper()
    if m == 'mape':
        print(f"{lb:<8} {mv:>9.2f}% {lo:>11.2f}% {hi:>11.2f}%")
    else:
        print(f"{lb:<8} {mv:>10.4f} {lo:>12.4f} {hi:>12.4f}")
print("="*70)

In [ ]:
from lightgbm import LGBMRegressor
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit
from scipy import stats

def mean_ci(vals, conf=0.95):
    vals = np.asarray(vals, dtype=float)
    vals = vals[~np.isnan(vals)]
    n = len(vals)
    if n < 2:
        return np.mean(vals), np.nan, np.nan
    m  = np.mean(vals)
    se = stats.sem(vals)
    h  = se * stats.t.ppf((1 + conf) / 2., n - 1)
    return m, m - h, m + h

train_size_lgbm = int(0.8 * len(X))
X_train_lgbm    = X.iloc[:train_size_lgbm]
y_train_lgbm    = y.iloc[:train_size_lgbm]

_tscv = TimeSeriesSplit(n_splits=10)
_fvl  = []

for fold, (tri, vli) in enumerate(_tscv.split(X_train_lgbm), 1):
    _Xtr, _Xvl = X_train_lgbm.iloc[tri], X_train_lgbm.iloc[vli]
    _ytr, _yvl = y_train_lgbm.iloc[tri], y_train_lgbm.iloc[vli]
    _md = LGBMRegressor(**grid_search.best_params_, random_state=42, n_jobs=-1, verbose=-1)
    _md.fit(_Xtr, _ytr)
    _vp  = _md.predict(_Xvl)
    _yv  = np.array(_yvl).flatten()
    _mv  = _yv != 0
    _fvl.append({'fold': fold,
                 'r2':   r2_score(_yv, _vp),
                 'mae':  mean_absolute_error(_yv, _vp),
                 'mape': np.mean(np.abs((_yv[_mv]-_vp[_mv])/_yv[_mv]))*100})
    print(f"  Fold {fold} — Val MAE: {_fvl[-1]['mae']:.4f}")

print()
print("="*70)
print("LIGHTGBM - 95% FOLD-LEVEL CI (VALIDATION)")
print("="*70)
print(f"{'Metric':<8} {'Mean':>10} {'CI Lower':>12} {'CI Upper':>12}")
print("-"*44)
for m in ['r2', 'mae', 'mape']:
    mv, lo, hi = mean_ci(pd.DataFrame(_fvl)[m])
    lb = m.upper()
    if m == 'mape':
        print(f"{lb:<8} {mv:>9.2f}% {lo:>11.2f}% {hi:>11.2f}%")
    else:
        print(f"{lb:<8} {mv:>10.4f} {lo:>12.4f} {hi:>12.4f}")
print("="*70)

lgbm_fvl = _fvl.copy()
lgbm_summary = three_way_table('LightGBM', y_train_lgbm, lgbm_train_pred, lgbm_fvl, y_test_lgbm, lgbm_test_pred)

In [ ]:
from sklearn.linear_model import Ridge

def direction_accuracy(y_true, y_pred, y_prev):
    true_dir = np.sign(np.asarray(y_true).ravel() - np.asarray(y_prev).ravel())
    pred_dir = np.sign(np.asarray(y_pred).ravel() - np.asarray(y_prev).ravel())
    return np.mean(true_dir == pred_dir)

def eval_metrics(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true).ravel(), np.asarray(y_pred).ravel()
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    nz   = y_true != 0
    mape = np.mean(np.abs((y_true[nz]-y_pred[nz])/y_true[nz]))*100
    return mae, rmse, r2, mape

def trading_simulation(y_true, y_pred, y_prev, cost_bps=10):
    """
    Simple long/flat strategy: go long when the model predicts the price
    will rise, otherwise stay in cash. Charges a transaction cost each
    time the position changes. cost_bps = cost per trade in basis points
    (10 bps = 0.10%).
    """
    y_true, y_pred, y_prev = (np.asarray(a).ravel() for a in (y_true, y_pred, y_prev))
    actual_return   = (y_true - y_prev) / y_prev
    position        = (y_pred > y_prev).astype(int)          # 1 = long, 0 = flat
    strategy_return = position * actual_return
    trades          = np.abs(np.diff(np.concatenate([[0], position])))
    cost            = trades * (cost_bps / 10000)
    net_return      = strategy_return - cost

    return {
        'Total Return':      np.prod(1 + net_return) - 1,
        'Buy & Hold Return': np.prod(1 + actual_return) - 1,
        'Num Trades':        int(trades.sum()),
    }

# Sanity check: confirm LightGBM's reloaded test set actually matches the
# shared X_test/y_test before comparing it in the same table.
assert np.allclose(y_test_lgbm.values, y_test.values), "LightGBM test set doesn't match shared test set — check the reload."

# --- Baselines, built once, scored on the SAME X_test/y_test as RF/ET/XGB/LightGBM ---
lag_col = 'price_lag1'
y_train_arr, y_test_arr = np.asarray(y_train).ravel(), np.asarray(y_test).ravel()

persistence_pred = X_test[lag_col].values
avg_change       = np.mean(np.diff(y_train_arr))
drift_pred       = X_test[lag_col].values + avg_change
hist_mean_pred   = np.full(len(y_test_arr), y_train_arr.mean())

price_lag_cols = [c for c in X_train.columns if 'price_lag' in c]
ar_model = Ridge(alpha=1.0).fit(X_train[price_lag_cols], y_train)
ar_pred  = ar_model.predict(X_test[price_lag_cols])

ridge_model = Ridge(alpha=1.0).fit(X_train, y_train)
ridge_pred  = ridge_model.predict(X_test)

baselines = {
    'Persistence': persistence_pred, 'Drift': drift_pred,
    'Historical Mean': hist_mean_pred, 'AR (price lags only)': ar_pred,
    'Ridge (full features)': ridge_pred,
}
model_preds = {
    'Random Forest': rf_test_pred, 'Extra Trees': et_test_pred,
    'XGBoost': xgb_test_pred, 'LightGBM': lgbm_test_pred,
}

# --- Accuracy table ---
rows = []
for name, preds in {**baselines, **model_preds}.items():
    mae, rmse, r2, mape = eval_metrics(y_test, preds)
    rows.append({'Model': name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape, 'R2': r2})
results_df = pd.DataFrame(rows).set_index('Model')

base_mae, base_rmse = results_df.loc['Persistence', ['MAE', 'RMSE']]
results_df['Relative MAE']  = results_df['MAE']  / base_mae
results_df['Relative RMSE'] = results_df['RMSE'] / base_rmse

for name, preds in {**baselines, **model_preds}.items():
    results_df.loc[name, 'Direction Accuracy'] = direction_accuracy(y_test, preds, X_test[lag_col].values)

print(results_df.round(4).to_string())

# --- Trading simulation ---
trade_rows = []
for name, preds in {**baselines, **model_preds}.items():
    res = trading_simulation(y_test, preds, X_test[lag_col].values)
    res['Model'] = name
    trade_rows.append(res)
trading_df = pd.DataFrame(trade_rows).set_index('Model')

print()
print(trading_df.round(4).to_string())

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files

In [ ]:
stock_dortmund = pd.read_csv("Borussia Dortmund Stock Price History.csv")
stock_dortmund["Date"] = pd.to_datetime(stock_dortmund["Date"])

stock_dortmund

In [ ]:
match_dortmund = pd.read_csv("dortmund_2000_to_2025.csv")

match_dortmund = match_dortmund[["hometeam", "awayteam", "homeelo", "awayelo", "ftresult", "matchdate"]]
match_dortmund["Date"] = pd.to_datetime(match_dortmund["matchdate"])

df_filtered = pd.DataFrame()

In [ ]:
#Date
df_filtered['Date'] = match_dortmund['Date']
#Result
dortmund_result = []
for i in range(len(match_dortmund)):
    if (match_dortmund['ftresult'].iloc[i] == 'A' and match_dortmund['hometeam'].iloc[i] == 'Dortmund') or \
       (match_dortmund['ftresult'].iloc[i] == 'H' and match_dortmund['awayteam'].iloc[i] == 'Dortmund'):
        dortmund_result.append(0)
    elif match_dortmund['ftresult'].iloc[i] == 'D':
        dortmund_result.append(1)
    else:
        dortmund_result.append(2)
df_filtered['dortmund_result'] = dortmund_result
#Elo
dortmund_elo = []
for i in range(len(match_dortmund)):
    if match_dortmund['hometeam'].iloc[i] == 'Dortmund':
        dortmund_elo.append(match_dortmund['homeelo'].iloc[i])
    else:
        dortmund_elo.append(match_dortmund['awayelo'].iloc[i])
df_filtered['dortmund_elo'] = dortmund_elo

df_filtered

In [ ]:
df_filtered['Date'] = pd.to_datetime(df_filtered['Date']).dt.date
stock_dortmund['Date'] = pd.to_datetime(stock_dortmund['Date']).dt.date
min_date = stock_dortmund['Date'].min()
max_date = max(df_filtered['Date'].max(), stock_dortmund['Date'].max())
all_dates = pd.date_range(start=min_date, end=max_date, freq='D')
combined_df = pd.DataFrame({'Date': all_dates.date})
combined_df = combined_df.merge(df_filtered, on='Date', how='left')
combined_df = combined_df.merge(stock_dortmund, on='Date', how='left')
combined_df['Price'] = combined_df['Price'].ffill()

df_clean = combined_df.dropna(subset=['dortmund_result']).reset_index(drop=True)

combined_df.head(60)

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import backend as K
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
import itertools
import time
import gc
import matplotlib.pyplot as plt

# --- Fix: pin down the random starting point so LSTM gives the same
# result every time this cell is run, instead of a different one each time. ---
tf.random.set_seed(42)
np.random.seed(42)

df_clean = df_clean.sort_values('Date').reset_index(drop=True)
feature_columns = ['Price', 'dortmund_result', 'dortmund_elo']
df_features = df_clean[feature_columns].dropna().copy()

if len(df_features) > 400:
    df_features = df_features.drop(index=df_features.index[400]).reset_index(drop=True)

scaler_X = MinMaxScaler()
df_features_scaled = pd.DataFrame(scaler_X.fit_transform(df_features), columns=feature_columns)

window_size = 4
X_windows   = []
y_values    = []

for i in range(len(df_features_scaled) - window_size):
    X_windows.append(df_features_scaled.iloc[i:i+window_size].values)
    y_values.append(df_features.iloc[i+window_size]['Price'])

X = np.array(X_windows)
y = np.array(y_values).reshape(-1, 1)

train_size = int(0.8 * len(X))
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

print(f"X_train shape: {X_train.shape} (Scaled)")
print(f"y_train shape: {y_train.shape} (Raw Prices)")

param_grid = {
    'architecture_type': ['bidirectional'],
    'num_layers':        [2],
    'units':             [64],
    'learning_rate':     [0.001],
    'dropout_rate':      [0.1]
}


def build_lstm_model(architecture_type, num_layers, units, learning_rate, dropout_rate, input_shape):
    model = Sequential()
    for i in range(num_layers):
        is_last           = (i == num_layers - 1)
        layer_input_shape = input_shape if i == 0 else None
        if architecture_type == 'standard':
            model.add(LSTM(units, activation='tanh', return_sequences=not is_last, input_shape=layer_input_shape))
        else:
            model.add(Bidirectional(LSTM(units, activation='tanh', return_sequences=not is_last), input_shape=layer_input_shape))
        if dropout_rate > 0:
            model.add(Dropout(dropout_rate))
    model.add(Dense(1, activation='linear'))
    model.compile(optimizer=Adam(learning_rate=learning_rate), loss='mse', metrics=['mae'])
    return model

tscv               = TimeSeriesSplit(n_splits=10)
param_combinations = list(itertools.product(*param_grid.values()))
results            = []
best_score         = float('inf')
best_params        = None
start_time         = time.time()

for idx, (arch, layers, units, lr, drop) in enumerate(param_combinations, 1):
    fold_scores = []
    for train_idx, val_idx in tscv.split(X_train):
        K.clear_session()
        X_f_train, X_f_val = X_train[train_idx], X_train[val_idx]
        y_f_train, y_f_val = y_train[train_idx], y_train[val_idx]
        model = build_lstm_model(arch, layers, units, lr, drop, (X.shape[1], X.shape[2]))
        stop  = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
        model.fit(X_f_train, y_f_train, epochs=50, batch_size=32, verbose=0,
                  callbacks=[stop], validation_data=(X_f_val, y_f_val))
        preds = model.predict(X_f_val, verbose=0)
        fold_scores.append(mean_absolute_error(y_f_val, preds))

    avg_mae = np.mean(fold_scores)
    results.append({'arch': arch, 'layers': layers, 'units': units, 'lr': lr, 'drop': drop, 'mae': avg_mae})
    print(f"Config {idx}/{len(param_combinations)} | {arch} | Layers: {layers} | MAE: {avg_mae:.4f}")

    if avg_mae < best_score:
        best_score  = avg_mae
        best_params = (arch, layers, units, lr, drop)
        K.clear_session()
        final_model = build_lstm_model(*best_params, (X.shape[1], X.shape[2]))
        final_model.fit(X_train, y_train, epochs=100, batch_size=32, verbose=0)
        lstm_test_pred = final_model.predict(X_test, verbose=0).flatten()

lstm_train_pred = final_model.predict(X_train, verbose=0).flatten()

lstm_train_r2   = r2_score(y_train, lstm_train_pred)
lstm_train_mae  = mean_absolute_error(y_train, lstm_train_pred)
lstm_train_rmse = np.sqrt(mean_squared_error(y_train, lstm_train_pred))
lstm_train_mape = np.mean(np.abs((y_train.flatten() - lstm_train_pred) / y_train.flatten())) * 100

lstm_test_r2   = r2_score(y_test, lstm_test_pred)
lstm_test_mae  = mean_absolute_error(y_test, lstm_test_pred)
lstm_test_rmse = np.sqrt(mean_squared_error(y_test, lstm_test_pred))
lstm_test_mape = np.mean(np.abs((y_test.flatten() - lstm_test_pred) / y_test.flatten())) * 100

print("\n" + "="*70)
print("LSTM - OVERALL RESULTS")
print("="*70)
print(f"{'Metric':<10} {'Train':>12} {'Test':>12}")
print("-"*36)
print(f"{'R²':<10} {lstm_train_r2:>12.4f} {lstm_test_r2:>12.4f}")
print(f"{'MAE':<10} {lstm_train_mae:>12.4f} {lstm_test_mae:>12.4f}")
print(f"{'RMSE':<10} {lstm_train_rmse:>12.4f} {lstm_test_rmse:>12.4f}")
print(f"{'MAPE':<10} {lstm_train_mape:>11.2f}% {lstm_test_mape:>11.2f}%")
print("="*70)

plt.figure(figsize=(10, 6))
plt.scatter(y_test, lstm_test_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.title(f"LSTM: Predicted vs Actual Price (Raw Scale)\nR²: {lstm_test_r2:.4f}")
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.show()

In [ ]:
import numpy as np
from sklearn.utils import resample
from sklearn.metrics import r2_score, mean_absolute_error

def bootstrap_ci(y_true, y_pred, n_iterations=1000, block_length=8, seed=42):
    """
    Moving-block bootstrap. Resamples contiguous blocks of `block_length`
    consecutive observations (with replacement) and concatenates them to
    rebuild a resampled series of the original length, preserving the
    autocorrelation between adjacent forecast errors that i.i.d. row
    resampling destroys.
    """
    rng = np.random.default_rng(seed)
    y_true = np.array(y_true).ravel()
    y_pred = np.array(y_pred).ravel()
    mask_nan = ~np.isnan(y_pred)
    y_true = y_true[mask_nan]
    y_pred = y_pred[mask_nan]
    n = len(y_true)
    n_blocks = int(np.ceil(n / block_length))

    st = {'r2': [], 'mae': [], 'mape': []}
    for _ in range(n_iterations):
        starts = rng.integers(0, n - block_length + 1, n_blocks)
        idx = np.concatenate([np.arange(s, s + block_length) for s in starts])[:n]
        y_t, y_p = y_true[idx], y_pred[idx]
        st['r2'].append(r2_score(y_t, y_p))
        st['mae'].append(mean_absolute_error(y_t, y_p))
        nz = y_t != 0
        st['mape'].append(np.mean(np.abs((y_t[nz]-y_p[nz])/y_t[nz]))*100 if nz.any() else np.nan)
    return st

_bt = bootstrap_ci(y_train, lstm_train_pred)
print("="*70)
print("LSTM - 95% BOOTSTRAP CI (TRAINING SET)")
print("="*70)
print(f"{'Metric':<8} {'Mean':>10} {'CI Lower':>12} {'CI Upper':>12}")
print("-"*44)
for m in ['r2', 'mae', 'mape']:
    mv = np.nanmean(_bt[m])
    lo = np.nanpercentile(_bt[m], 2.5)
    hi = np.nanpercentile(_bt[m], 97.5)
    lb = m.upper()
    if m == 'mape':
        print(f"{lb:<8} {mv:>9.2f}% {lo:>11.2f}% {hi:>11.2f}%")
    else:
        print(f"{lb:<8} {mv:>10.4f} {lo:>12.4f} {hi:>12.4f}")
print("="*70)

_bt2 = bootstrap_ci(y_test, lstm_test_pred)
print()
print("="*70)
print("LSTM - 95% BOOTSTRAP CI (TEST SET)")
print("="*70)
print(f"{'Metric':<8} {'Mean':>10} {'CI Lower':>12} {'CI Upper':>12}")
print("-"*44)
for m in ['r2', 'mae', 'mape']:
    mv = np.nanmean(_bt2[m])
    lo = np.nanpercentile(_bt2[m], 2.5)
    hi = np.nanpercentile(_bt2[m], 97.5)
    lb = m.upper()
    if m == 'mape':
        print(f"{lb:<8} {mv:>9.2f}% {lo:>11.2f}% {hi:>11.2f}%")
    else:
        print(f"{lb:<8} {mv:>10.4f} {lo:>12.4f} {hi:>12.4f}")
print("="*70)

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit
from scipy import stats

# --- Fix: same reason as Cell 57 — this cell also builds fresh LSTMs
# from scratch (once per fold), so it needs its own seed too. ---
tf.random.set_seed(42)
np.random.seed(42)

def mean_ci(vals, conf=0.95):
    vals = np.asarray(vals, dtype=float)
    vals = vals[~np.isnan(vals)]
    n = len(vals)
    if n < 2:
        return np.mean(vals), np.nan, np.nan
    m  = np.mean(vals)
    se = stats.sem(vals)
    h  = se * stats.t.ppf((1 + conf) / 2., n - 1)
    return m, m - h, m + h

split_idx    = int(0.8 * len(X))
X_train_lstm = X[:split_idx]
y_train_lstm = y[:split_idx]

tscv_lstm           = TimeSeriesSplit(n_splits=10)
lstm_fold_train_res = []
lstm_fold_val_res   = []

print("Starting LSTM Fold-Level Evaluation (This may take a few minutes)...")

for fold, (train_idx, val_idx) in enumerate(tscv_lstm.split(X_train_lstm), 1):
    X_f_train, X_f_val = X_train_lstm[train_idx], X_train_lstm[val_idx]
    y_f_train, y_f_val = y_train_lstm[train_idx], y_train_lstm[val_idx]

    scaler_x = MinMaxScaler()
    scaler_y = MinMaxScaler()

    X_f_train_2d = X_f_train.reshape(X_f_train.shape[0], -1)
    X_f_val_2d   = X_f_val.reshape(X_f_val.shape[0], -1)

    X_f_train_sc = scaler_x.fit_transform(X_f_train_2d)
    X_f_val_sc   = scaler_x.transform(X_f_val_2d)
    y_f_train_sc = scaler_y.fit_transform(y_f_train.reshape(-1, 1))

    X_f_train_3d = X_f_train_sc.reshape((X_f_train_sc.shape[0], 1, X_f_train_sc.shape[1]))
    X_f_val_3d   = X_f_val_sc.reshape((X_f_val_sc.shape[0],   1, X_f_val_sc.shape[1]))

    model = Sequential([
        LSTM(64, activation='relu', input_shape=(1, X_f_train_sc.shape[1]), return_sequences=False),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse')
    model.fit(X_f_train_3d, y_f_train_sc, epochs=50, batch_size=32, verbose=0)

    tr_preds_sc = model.predict(X_f_train_3d, verbose=0)
    tr_preds    = scaler_y.inverse_transform(tr_preds_sc).flatten()
    y_tr_true   = y_f_train.flatten()
    mask_tr     = y_tr_true != 0
    lstm_fold_train_res.append({
        'fold': fold,
        'r2':   r2_score(y_tr_true, tr_preds),
        'mae':  mean_absolute_error(y_tr_true, tr_preds),
        'mape': np.mean(np.abs((y_tr_true[mask_tr]-tr_preds[mask_tr])/y_tr_true[mask_tr]))*100
    })

    val_preds_sc = model.predict(X_f_val_3d, verbose=0)
    val_preds    = scaler_y.inverse_transform(val_preds_sc).flatten()
    y_val_true   = y_f_val.flatten()
    safe_denom   = np.where(y_val_true == 0, 1, y_val_true)
    lstm_fold_val_res.append({
        'fold': fold,
        'r2':   r2_score(y_val_true, val_preds),
        'mae':  mean_absolute_error(y_val_true, val_preds),
        'mape': np.mean(np.abs((y_val_true - val_preds) / safe_denom)) * 100
    })
    print(f"  Fold {fold} — Train MAE: {lstm_fold_train_res[-1]['mae']:.4f} | Val MAE: {lstm_fold_val_res[-1]['mae']:.4f}")

for _lbl, _df in [("LSTM - 95% FOLD-LEVEL CI (TRAINING FOLDS)", pd.DataFrame(lstm_fold_train_res)),
                   ("LSTM - 95% FOLD-LEVEL CI (VALIDATION FOLDS)", pd.DataFrame(lstm_fold_val_res))]:
    print()
    print("="*70)
    print(_lbl)
    print("="*70)
    print(f"{'Metric':<8} {'Mean':>10} {'CI Lower':>12} {'CI Upper':>12}")
    print("-"*44)
    for m in ['r2', 'mae', 'mape']:
        mv, lo, hi = mean_ci(_df[m])
        lb = m.upper()
        if m == 'mape':
            print(f"{lb:<8} {mv:>9.2f}% {lo:>11.2f}% {hi:>11.2f}%")
        else:
            print(f"{lb:<8} {mv:>10.4f} {lo:>12.4f} {hi:>12.4f}")
    print("="*70)

lstm_summary = three_way_table('LSTM', y_train, lstm_train_pred, lstm_fold_val_res, y_test, lstm_test_pred)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import mean_absolute_error

feature_names = [f"{feat}_t-{window_size - t}" for t in range(window_size) for feat in feature_columns]
X_test_flat   = X_test.reshape(X_test.shape[0], -1).astype(np.float32)

def predict(X):
    return tf.function(final_model)(tf.constant(X, dtype=tf.float32)).numpy().flatten()

baseline_mae = mean_absolute_error(y_test, predict(X_test))

importance_scores = []
for i, fname in enumerate(feature_names):
    X_perm = X_test_flat.copy()
    X_perm[:, i] = np.random.permutation(X_perm[:, i])
    mae = mean_absolute_error(y_test, predict(X_perm.reshape(-1, window_size, len(feature_columns))))
    importance_scores.append(mae - baseline_mae)
    print(f"{fname}: {importance_scores[-1]:.4f}")

importance_df = pd.DataFrame({'feature': feature_names, 'importance': importance_scores})

importance_df.sort_values('importance', ascending=True).plot(
    kind='barh', x='feature', y='importance', legend=False, figsize=(8, 7))
plt.title("Feature Importance (MAE increase when permuted)")
plt.tight_layout()
plt.show()

agg = {feat: importance_df[importance_df['feature'].str.startswith(feat)]['importance'].sum()
       for feat in feature_columns}
pd.DataFrame.from_dict(agg, orient='index', columns=['importance']) \
  .sort_values('importance', ascending=True) \
  .plot(kind='barh', legend=False)
plt.title("Importance by Variable (all lags combined)")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, len(feature_columns), figsize=(14, 4))
for ax, feat in zip(axes, feature_columns):
    sub = importance_df[importance_df['feature'].str.startswith(feat)]
    ax.bar(sub['feature'], sub['importance'])
    ax.set_title(feat)
    ax.tick_params(axis='x', rotation=45)
plt.suptitle("Importance by Lag")
plt.tight_layout()
plt.show()

In [ ]:
def direction_accuracy(y_true, y_pred, y_prev):
    """Direction = did the price go up or down relative to the last known price."""
    true_dir = np.sign(np.asarray(y_true).ravel() - np.asarray(y_prev).ravel())
    pred_dir = np.sign(np.asarray(y_pred).ravel() - np.asarray(y_prev).ravel())
    return np.mean(true_dir == pred_dir)

def trading_simulation(y_true, y_pred, y_prev, cost_bps=10):
    """
    Long when the model predicts the price will rise relative to the last
    known price, flat otherwise. Transaction cost charged on each position
    change.
    """
    y_true, y_pred, y_prev = (np.asarray(a).ravel() for a in (y_true, y_pred, y_prev))
    actual_return   = (y_true - y_prev) / y_prev
    position        = (y_pred > y_prev).astype(int)
    strategy_return = position * actual_return
    trades          = np.abs(np.diff(np.concatenate([[0], position])))
    cost            = trades * (cost_bps / 10000)
    net_return      = strategy_return - cost
    return {
        'Total Return':      np.prod(1 + net_return) - 1,
        'Buy & Hold Return': np.prod(1 + actual_return) - 1,
        'Num Trades':        int(trades.sum()),
    }

lstm_y_train_arr, lstm_y_test_arr = y_train.ravel(), y_test.ravel()

# --- Recover the REAL previous price (this is the fix) ---
# X_test holds SCALED features (0-1 range) since it was built from
# df_features_scaled. Grabbing X_test[:, -1, 0] directly gives you the
# scaled price, not the real one — comparing it to y_test (a real price)
# is meaningless. Inverse-transform the last timestep back to real units first.
last_step_scaled = X_test[:, -1, :]                      # shape (n, 3): [Price, result, elo], scaled
last_step_raw    = scaler_X.inverse_transform(last_step_scaled)
lstm_lag1_price  = last_step_raw[:, feature_columns.index('Price')]   # real "previous price" per row

# --- Baselines, price-level target ---
lstm_persistence_pred = lstm_lag1_price
lstm_avg_change       = np.mean(np.diff(lstm_y_train_arr))
lstm_drift_pred       = lstm_lag1_price + lstm_avg_change
lstm_hist_mean_pred   = np.full(len(lstm_y_test_arr), lstm_y_train_arr.mean())

X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_test_flat  = X_test.reshape(X_test.shape[0], -1)
price_cols   = list(range(0, X_train_flat.shape[1], len(feature_columns)))
lstm_ar_pred = Ridge(alpha=1.0).fit(X_train_flat[:, price_cols], lstm_y_train_arr) \
                                 .predict(X_test_flat[:, price_cols])
lstm_ridge_pred = Ridge(alpha=1.0).fit(X_train_flat, lstm_y_train_arr).predict(X_test_flat)

lstm_baselines = {
    'Persistence':             lstm_persistence_pred,
    'Drift':                   lstm_drift_pred,
    'Historical Mean':         lstm_hist_mean_pred,
    'AR (price channel only)': lstm_ar_pred,
    'Ridge (full features)':   lstm_ridge_pred,
}

# --- Accuracy table ---
lstm_rows = []
for name, preds in {**lstm_baselines, 'LSTM': lstm_test_pred}.items():
    mae, rmse, r2, mape = eval_metrics(lstm_y_test_arr, preds)
    lstm_rows.append({'Model': name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape, 'R2': r2})
lstm_results_df = pd.DataFrame(lstm_rows).set_index('Model')

base_mae, base_rmse = lstm_results_df.loc['Persistence', ['MAE', 'RMSE']]
lstm_results_df['Relative MAE']  = lstm_results_df['MAE']  / base_mae
lstm_results_df['Relative RMSE'] = lstm_results_df['RMSE'] / base_rmse

for name, preds in {**lstm_baselines, 'LSTM': lstm_test_pred}.items():
    lstm_results_df.loc[name, 'Direction Accuracy'] = direction_accuracy(lstm_y_test_arr, preds, lstm_lag1_price)

print(lstm_results_df.round(4).to_string())

# --- Trading simulation ---
lstm_trade_rows = []
for name, preds in {**lstm_baselines, 'LSTM': lstm_test_pred}.items():
    res = trading_simulation(lstm_y_test_arr, preds, lstm_lag1_price)
    res['Model'] = name
    lstm_trade_rows.append(res)
lstm_trading_df = pd.DataFrame(lstm_trade_rows).set_index('Model')

print()
print(lstm_trading_df.round(4).to_string())

In [ ]:
import numpy as np
import pandas as pd
from itertools import combinations
from scipy import stats

def dm_test(actual, pred1, pred2, loss='squared', h=1):
    """
    Diebold-Mariano test with Harvey-Leybourne-Newbold (HLN) small-sample correction.
    loss: 'squared' or 'absolute'
    h:    forecast horizon (1 for one-step-ahead)
    Positive DM stat means Model 1 has larger errors (Model 2 is better).
    Negative DM stat means Model 1 has smaller errors (Model 1 is better).
    """
    actual = np.array(actual).ravel()
    pred1  = np.array(pred1).ravel()
    pred2  = np.array(pred2).ravel()

    n = min(len(actual), len(pred1), len(pred2))
    actual, pred1, pred2 = actual[-n:], pred1[-n:], pred2[-n:]

    if loss == 'squared':
        e1 = (actual - pred1) ** 2
        e2 = (actual - pred2) ** 2
    elif loss == 'absolute':
        e1 = np.abs(actual - pred1)
        e2 = np.abs(actual - pred2)
    else:
        raise ValueError("loss must be 'squared' or 'absolute'")

    d     = e1 - e2
    T     = len(d)
    d_bar = np.mean(d)

    gamma0    = np.var(d, ddof=0)
    gamma_sum = 0.0
    for k in range(1, h):
        gamma_k    = np.mean((d[k:] - d_bar) * (d[:-k] - d_bar))
        gamma_sum += (1 - k / h) * gamma_k
    var_d_bar = (gamma0 + 2 * gamma_sum) / T

    if var_d_bar <= 0:
        return np.nan, np.nan

    dm_stat     = d_bar / np.sqrt(var_d_bar)
    hln_factor  = np.sqrt((T + 1 - 2*h + h*(h-1)/T) / T)
    dm_stat_hln = dm_stat * hln_factor
    p_value     = 2 * stats.t.sf(np.abs(dm_stat_hln), df=T - 1)

    return dm_stat_hln, p_value


y_actual = np.array(y_test_lgbm).ravel()

model_test_preds = {
    'RandomForest': np.array(rf_test_pred).ravel()    if 'rf_test_pred'   in dir() else None,
    'ExtraTrees':   np.array(et_test_pred).ravel()    if 'et_test_pred'   in dir() else None,
    'XGBoost':      np.array(xgb_test_pred).ravel()   if 'xgb_test_pred'  in dir() else None,
    'LSTM':         np.array(lstm_test_pred).ravel()  if 'lstm_test_pred' in dir() else None,
    'LightGBM':     np.array(lgbm_test_pred).ravel()  if 'lgbm_test_pred' in dir() else None,
}

model_test_preds = {k: v for k, v in model_test_preds.items() if v is not None}
model_names      = list(model_test_preds.keys())
pairs            = list(combinations(model_names, 2))

print(f"Models loaded: {model_names}")
print(f"Total pairwise comparisons: {len(pairs)}")

results_sq  = []
results_abs = []

for m1, m2 in pairs:
    p1 = model_test_preds[m1]
    p2 = model_test_preds[m2]

    dm_sq,  p_sq  = dm_test(y_actual, p1, p2, loss='squared',  h=1)
    dm_abs, p_abs = dm_test(y_actual, p1, p2, loss='absolute', h=1)

    results_sq.append({
        'Model 1':      m1,
        'Model 2':      m2,
        'DM Stat':      round(dm_sq,  4) if not np.isnan(dm_sq)  else 'NaN',
        'p-value':      round(p_sq,   4) if not np.isnan(p_sq)   else 'NaN',
        'Sig (p<0.05)': '✓' if (not np.isnan(p_sq)  and p_sq  < 0.05) else '✗',
        'Better Model': m1 if (not np.isnan(dm_sq)  and dm_sq  > 0) else m2
    })

    results_abs.append({
        'Model 1':      m1,
        'Model 2':      m2,
        'DM Stat':      round(dm_abs, 4) if not np.isnan(dm_abs) else 'NaN',
        'p-value':      round(p_abs,  4) if not np.isnan(p_abs)  else 'NaN',
        'Sig (p<0.05)': '✓' if (not np.isnan(p_abs) and p_abs < 0.05) else '✗',
        'Better Model': m1 if (not np.isnan(dm_abs) and dm_abs > 0) else m2
    })

df_dm_sq  = pd.DataFrame(results_sq)
df_dm_abs = pd.DataFrame(results_abs)

print("\n" + "="*90)
print("DIEBOLD-MARIANO TEST (HLN-CORRECTED) — SQUARED ERROR LOSS")
print("="*90)
print("Interpretation: DM Stat > 0 → Model 1 has LARGER squared errors (Model 2 is better).")
print("               DM Stat < 0 → Model 1 has SMALLER squared errors (Model 1 is better).")
print("-"*90)
print(df_dm_sq.to_string(index=False))
print("="*90)

print()
print("="*90)
print("DIEBOLD-MARIANO TEST (HLN-CORRECTED) — ABSOLUTE ERROR LOSS")
print("="*90)
print("Interpretation: DM Stat > 0 → Model 1 has LARGER absolute errors (Model 2 is better).")
print("               DM Stat < 0 → Model 1 has SMALLER absolute errors (Model 1 is better).")
print("-"*90)
print(df_dm_abs.to_string(index=False))
print("="*90)

print("\nNote: HLN correction applied. p-values from t-distribution with T-1 degrees of freedom.")
print("Note: ✓ = statistically significant difference at the 5% level.")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.model_selection import TimeSeriesSplit
import pandas as pd

colors = {
    'Random Forest': 'orange',
    'Extra Trees':   'purple',
    'XGBoost':       'green',
    'LightGBM':      'blue',
    'LSTM':          'red'
}

rf_mapes   = [d['mape'] for d in rf_fvl]
et_mapes   = [d['mape'] for d in et_fvl]
xgb_mapes  = [d['mape'] for d in xgb_fvl]
lgbm_mapes = [d['mape'] for d in lgbm_fvl]
lstm_mapes = [d['mape'] for d in lstm_fold_val_res]

# --- Build the real validation-period start date for each of the 10 folds,
# inlined here so this cell doesn't depend on a separate script having run ---
train_size = int(0.8 * len(df_model))
train_dates = df_model['Date'].iloc[:train_size].reset_index(drop=True)

tscv = TimeSeriesSplit(n_splits=10)
val_start_dates = []
for train_idx, val_idx in tscv.split(train_dates):
    val_start_dates.append(train_dates.iloc[val_idx[0]])

fold_labels = [pd.Timestamp(d).strftime('%Y-%m') for d in val_start_dates]
folds = list(range(1, 11))

fig, ax1 = plt.subplots(figsize=(14, 7))

for name, mapes in [('Random Forest', rf_mapes), ('Extra Trees', et_mapes),
                     ('XGBoost', xgb_mapes), ('LightGBM', lgbm_mapes),
                     ('LSTM', lstm_mapes)]:
    ax1.plot(folds, mapes, color=colors[name], marker='o', linewidth=2, label=name)

ax1.set_xlabel('Validation Period Start (Year-Month)', fontsize=14)
ax1.set_ylabel('MAPE (%)', fontsize=14)
ax1.set_xticks(folds)
ax1.set_xticklabels(fold_labels, fontsize=11, rotation=45)
ax1.tick_params(axis='y', labelsize=12)
ax1.set_title('MAPE Across CV Validation Folds — All Models', fontsize=15, fontweight='bold')
ax1.legend(fontsize=12, loc='upper left')
ax1.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
all_summaries_df = pd.DataFrame([rf_summary, et_summary, xgb_summary, lgbm_summary, lstm_summary])
print(all_summaries_df.round(4).to_string(index=False))